# Aplikasi Prediksi Kunjungan Wisatawan Kabupaten Bangkalan

In [ ]:
!pip install streamlit folium plotly vmdpy EMD-signal joblib scikit-learn openpyxl pillow requests -q

In [27]:
# Jangan hapus cache_hasil agar hasil komputasi sebelumnya dapat langsung dimuat instan
!mkdir -p assets cache_hasil

# FORECAST ENGINE USER

In [28]:
%%writefile forecasting_engine.py
# -*- coding: utf-8 -*-
"""
forecasting_engine.py
Mesin peramalan kunjungan wisatawan untuk dashboard User.

Alur (mengikuti notebook skripsi, TANPA pencarian parameter):
  dataset -> pembersihan (0 -> interpolasi linear) -> dekomposisi sinyal
  -> normalisasi Min-Max per komponen -> sliding window 12 bulan
  -> split 80:20 -> latih model GRU per komponen memakai hyperparameter tetap
     (units, dropout, learning rate, batch size) + EarlyStopping seperti notebook
  -> evaluasi data uji (MAE, MAPE) -> prediksi rekursif 12 bulan ke depan.

Destinasi lain  : ICEEMDAN -> VMD -> GRU
Gunung Geger    : VMD -> GRU

CATATAN PERBAIKAN (v1.1) - penyebab hasil Streamlit berbeda dengan notebook:
  Di notebook (fungsi retrain_component) model GRU dilatih dengan
      validation_data = 10% data latih paling akhir,
      epochs = 200, EarlyStopping(monitor='val_loss', patience=15,
      restore_best_weights=True)
  sehingga bobot akhir = bobot pada epoch dengan val_loss terendah.
  Versi lama Streamlit melatih model SELAMA 'BestEpoch' epoch tanpa validasi dan
  tanpa restore_best_weights. 'BestEpoch' hanyalah hasil satu kali pelatihan acak
  di notebook; dengan inisialisasi bobot berbeda, epoch itu tidak lagi optimal
  (terutama untuk komponen dengan best epoch kecil / dropout besar), dan error
  dari 33 komponen terjumlah pada rekonstruksi akhir -> MAE/MAPE membengkak.
  Sekarang prosedur pelatihannya identik dengan notebook.
"""
import os
import re
import io
import gc
import json
import hashlib
import warnings
import gzip
import base64

import numpy as np
import pandas as pd

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
warnings.filterwarnings("ignore")

ENGINE_VERSION = "2.0"   # dinaikkan agar cache hasil lama (v1.0) tidak dipakai lagi
WINDOW = 12           # lag 12 bulan
TEST_RATIO = 0.2      # split 80:20
VAL_SPLIT = 0.10      # validasi internal 10% (sama seperti retrain di notebook)
MAX_EPOCHS = 200      # sama dengan notebook (max_epochs=200)
PATIENCE = 15         # sama dengan notebook (patience=15)
HORIZON = 12          # prediksi 12 bulan ke depan
MIN_ROWS = 48         # minimal data agar window + split masuk akal
CACHE_DIR = "cache_hasil"

# ============================================================
# PARAMETER GRU PER DESTINASI (ringkasan dari notebook)
# Kolom: No | Component | Units | Dropout | LearningRate | BatchSize | BestEpoch
# (BestEpoch hanya informasi; pelatihan memakai EarlyStopping seperti notebook)
# ============================================================
TBL_SYAIKHONA = """
1    IMF_1_VMD_1    8    0.251836    0.003    4    53
2    IMF_1_VMD_2    64    0.013434    0.003    4    45
3    IMF_1_VMD_3    64    0.048302    0.003    4    36
4    IMF_1_VMD_4    64    0.397597    0.003    4    8
5    IMF_1_VMD_5    64    0.002640    0.001    24    142
6    IMF_1_VMD_6    64    0.125742    0.001    8    28
7    IMF_1_VMD_7    16    0.107215    0.003    16    36
8    IMF_1_VMD_8    64    0.092664    0.001    4    28
9    IMF_1_VMD_9    64    0.001267    0.001    4    28
10    IMF_1_VMD_10    64    0.081237    0.003    4    20
11    IMF_1_VMD_11    16    0.028247    0.001    4    41
12    IMF_1_VMD_12    64    0.125742    0.001    8    89
13    IMF_2_VMD_1    16    0.166964    0.003    20    53
14    IMF_2_VMD_2    8    0.237610    0.003    4    33
15    IMF_2_VMD_3    64    0.204000    0.003    24    60
16    IMF_2_VMD_4    16    0.202958    0.003    4    20
17    IMF_2_VMD_5    16    0.269953    0.003    24    38
18    IMF_2_VMD_6    8    0.234958    0.003    4    36
19    IMF_2_VMD_7    16    0.243790    0.003    4    50
20    IMF_2_VMD_8    64    0.066046    0.001    4    24
21    IMF_2_VMD_9    64    0.220686    0.003    8    32
22    IMF_2_VMD_10    8    0.237610    0.003    4    44
23    IMF_2_VMD_11    16    0.166964    0.003    20    47
24    IMF_2_VMD_12    8    0.241461    0.003    4    48
25    IMF_3_VMD_1    16    0.253402    0.003    4    38
26    IMF_3_VMD_2    64    0.155471    0.003    8    13
27    IMF_3_VMD_3    64    0.119301    0.003    4    21
28    IMF_3_VMD_4    64    0.395587    0.003    4    16
29    IMF_4_VMD_1    64    0.155471    0.003    8    32
30    IMF_4_VMD_2    16    0.182617    0.003    4    26
31    IMF_5_VMD_1    64    0.058402    0.001    8    29
32    IMF_5_VMD_2    64    0.003523    0.003    4    17
33    ICEEMDAN_Residue    8    0.398927    0.001    4    31
"""

TBL_LANGGUNDI = """
1    IMF_1_VMD_1    64    0.002753    0.0010    4    31
2    IMF_1_VMD_2    64    0.125742    0.0010    8    112
3    IMF_1_VMD_3    64    0.125742    0.0010    8    38
4    IMF_1_VMD_4    64    0.006214    0.0030    4    41
5    IMF_1_VMD_5    64    0.067409    0.0010    16    45
6    IMF_1_VMD_6    8    0.241405    0.0030    4    100
7    IMF_1_VMD_7    64    0.155471    0.0030    8    33
8    IMF_1_VMD_8    64    0.007364    0.0010    4    51
9    IMF_1_VMD_9    64    0.004065    0.0010    4    38
10    IMF_1_VMD_10    8    0.237610    0.0030    4    30
11    IMF_1_VMD_11    64    0.119301    0.0030    4    42
12    IMF_1_VMD_12    64    0.155471    0.0030    8    42
13    IMF_2_VMD_1    64    0.119301    0.0030    4    25
14    IMF_2_VMD_2    16    0.182617    0.0030    4    9
15    IMF_2_VMD_3    64    0.125611    0.0030    4    38
16    IMF_2_VMD_4    64    0.050242    0.0010    16    61
17    IMF_2_VMD_5    64    0.004079    0.0030    4    11
18    IMF_2_VMD_6    64    0.155471    0.0030    8    16
19    IMF_2_VMD_7    64    0.342506    0.0030    4    39
20    IMF_2_VMD_8    64    0.155471    0.0030    8    20
21    IMF_2_VMD_9    64    0.155471    0.0030    8    21
22    IMF_3_VMD_1    8    0.214250    0.0030    20    60
23    IMF_3_VMD_2    64    0.003327    0.0010    4    65
24    IMF_3_VMD_3    64    0.155471    0.0030    8    23
25    IMF_3_VMD_4    64    0.002637    0.0010    4    47
26    IMF_3_VMD_5    64    0.005079    0.0030    4    51
27    IMF_3_VMD_6    16    0.062407    0.0003    16    170
28    IMF_3_VMD_7    16    0.166964    0.0030    20    66
29    IMF_3_VMD_8    16    0.239530    0.0030    4    35
30    IMF_3_VMD_9    8    0.211516    0.0030    4    5
31    IMF_3_VMD_10    64    0.396403    0.0030    4    17
32    IMF_3_VMD_11    64    0.264818    0.0030    8    55
33    IMF_3_VMD_12    64    0.003523    0.0030    4    25
34    IMF_4_VMD_1    64    0.172778    0.0003    8    91
35    IMF_4_VMD_2    64    0.155471    0.0030    8    21
36    ICEEMDAN_Residue    64    0.155471    0.0030    8    30
"""

TBL_CENDANA = """
1    IMF_1_VMD_1    64    0.012867    0.001    4    11
2    IMF_1_VMD_2    24    0.024561    0.003    4    72
3    IMF_1_VMD_3    8    0.237610    0.003    4    53
4    IMF_1_VMD_4    64    0.004079    0.003    4    108
5    IMF_1_VMD_5    16    0.211112    0.003    4    49
6    IMF_1_VMD_6    16    0.196056    0.003    20    87
7    IMF_1_VMD_7    64    0.110535    0.001    8    83
8    IMF_1_VMD_8    64    0.155471    0.003    8    69
9    IMF_1_VMD_9    64    0.155471    0.003    8    27
10    IMF_2_VMD_1    24    0.014495    0.003    4    125
11    IMF_2_VMD_2    64    0.294414    0.003    4    60
12    IMF_2_VMD_3    64    0.110535    0.001    8    64
13    IMF_2_VMD_4    64    0.155471    0.003    8    18
14    IMF_2_VMD_5    64    0.215504    0.003    4    71
15    IMF_2_VMD_6    64    0.155471    0.003    8    56
16    IMF_2_VMD_7    8    0.099316    0.003    16    7
17    IMF_2_VMD_8    16    0.212856    0.003    4    12
18    IMF_2_VMD_9    64    0.086446    0.003    4    78
19    IMF_2_VMD_10    64    0.155471    0.003    8    21
20    IMF_2_VMD_11    64    0.223634    0.003    4    52
21    IMF_3_VMD_1    16    0.239530    0.003    4    37
22    IMF_3_VMD_2    16    0.166964    0.003    20    55
23    IMF_3_VMD_3    64    0.125742    0.001    8    31
24    IMF_3_VMD_4    64    0.155471    0.003    8    22
25    IMF_3_VMD_5    64    0.155471    0.003    8    35
26    IMF_3_VMD_6    64    0.081270    0.003    4    16
27    IMF_3_VMD_7    64    0.013434    0.003    4    27
28    IMF_3_VMD_8    64    0.002925    0.003    4    22
29    IMF_3_VMD_9    8    0.237610    0.003    4    64
30    IMF_3_VMD_10    8    0.214250    0.003    4    35
31    IMF_3_VMD_11    64    0.004344    0.003    4    11
32    IMF_3_VMD_12    64    0.155471    0.003    8    25
33    IMF_4_VMD_1    64    0.155471    0.003    8    17
34    IMF_4_VMD_2    64    0.155471    0.003    8    29
35    IMF_4_VMD_3    64    0.189713    0.003    4    32
36    ICEEMDAN_Residue    16    0.120033    0.003    24    11
"""

TBL_AIRMATAIBU = """
1    IMF_1_VMD_1    16    0.000891    0.0030    16    19
2    IMF_1_VMD_2    64    0.155471    0.0030    8    28
3    IMF_1_VMD_3    64    0.002650    0.0030    4    19
4    IMF_1_VMD_4    16    0.166964    0.0030    20    97
5    IMF_1_VMD_5    64    0.155471    0.0030    8    28
6    IMF_1_VMD_6    16    0.013434    0.0030    24    62
7    IMF_1_VMD_7    8    0.321469    0.0030    24    140
8    IMF_1_VMD_8    8    0.237610    0.0030    4    81
9    IMF_1_VMD_9    64    0.155471    0.0030    8    23
10    IMF_1_VMD_10    64    0.391364    0.0030    20    26
11    IMF_1_VMD_11    64    0.172778    0.0003    8    90
12    IMF_1_VMD_12    16    0.214250    0.0030    16    37
13    IMF_2_VMD_1    8    0.234078    0.0010    4    69
14    IMF_2_VMD_2    16    0.088774    0.0030    4    42
15    IMF_2_VMD_3    64    0.029306    0.0030    4    20
16    IMF_2_VMD_4    64    0.155471    0.0030    8    76
17    IMF_3_VMD_1    16    0.166964    0.0030    20    4
18    IMF_3_VMD_2    64    0.155471    0.0030    8    15
19    IMF_3_VMD_3    16    0.234725    0.0030    4    21
20    IMF_3_VMD_4    64    0.006214    0.0030    4    28
21    IMF_3_VMD_5    16    0.166964    0.0030    20    40
22    IMF_3_VMD_6    64    0.013434    0.0030    4    31
23    IMF_3_VMD_7    64    0.155471    0.0030    8    22
24    IMF_4_VMD_1    64    0.125742    0.0010    8    7
25    IMF_4_VMD_2    16    0.062444    0.0010    4    38
26    IMF_4_VMD_3    64    0.282743    0.0001    8    60
27    IMF_4_VMD_4    16    0.166964    0.0030    20    62
28    IMF_5_VMD_1    64    0.155471    0.0030    8    1
29    IMF_5_VMD_2    64    0.092601    0.0010    8    11
30    IMF_5_VMD_3    8    0.237610    0.0030    4    19
31    IMF_5_VMD_4    64    0.155471    0.0030    8    14
32    IMF_5_VMD_5    24    0.061246    0.0030    4    16
33    IMF_5_VMD_6    64    0.000807    0.0030    4    17
34    IMF_5_VMD_7    64    0.397646    0.0030    4    59
35    IMF_5_VMD_8    8    0.271146    0.0030    24    1
36    IMF_5_VMD_9    64    0.080314    0.0010    8    43
37    IMF_5_VMD_10    64    0.155471    0.0030    8    37
38    IMF_5_VMD_11    16    0.166964    0.0030    20    7
39    IMF_5_VMD_12    64    0.357255    0.0030    4    29
40    IMF_6_VMD_1    8    0.143497    0.0030    4    50
41    IMF_6_VMD_2    64    0.155471    0.0030    8    1
42    ICEEMDAN_Residue    64    0.189713    0.0030    4    15
"""

# Gunung Geger: VMD-GRU saja (12 mode VMD, K=12, alpha=100 pada notebook)
TBL_GEGER = """
1    VMD_1    64    0.155471    0.0030    8    64
2    VMD_2    8    0.211112    0.0030    4    17
3    VMD_3    16    0.068906    0.0003    16    146
4    VMD_4    64    0.155471    0.0030    8    71
5    VMD_5    8    0.321469    0.0030    24    61
6    VMD_6    64    0.172778    0.0003    8    86
7    VMD_7    64    0.172778    0.0003    8    96
8    VMD_8    64    0.155471    0.0030    8    42
9    VMD_9    8    0.214250    0.0030    4    100
10    VMD_10    16    0.214250    0.0030    16    101
11    VMD_11    8    0.234078    0.0010    4    66
12    VMD_12    64    0.155471    0.0030    8    29
"""

# alpha VMD per IMF yang sudah tercatat di notebook (hasil optimal).
# Destinasi yang alpha-nya tidak diisi di sini dihitung otomatis dengan
# aturan yang sama seperti notebook (lihat _auto_alpha) untuk K yang sudah tetap.
CONFIG = {
    "Makam Syaikhona Kholil": {
        "metode": "hybrid", "tabel": TBL_SYAIKHONA,
        "alpha": {"IMF_1": 100, "IMF_2": 200, "IMF_3": 300, "IMF_4": 2000, "IMF_5": 2000},
    },
    "Kolla Langgundi": {
        "metode": "hybrid", "tabel": TBL_LANGGUNDI,
        "alpha": {"IMF_1": 100, "IMF_2": 1700, "IMF_3": 400, "IMF_4": 2000},
    },
    "Makam Sunan Cendana": {
        "metode": "hybrid", "tabel": TBL_CENDANA,
        "alpha": {"IMF_1": 100, "IMF_2": 300, "IMF_3": 200, "IMF_4": 2000},
    },
    "Makam Air Mata Ibu": {
        "metode": "hybrid", "tabel": TBL_AIRMATAIBU,
        "alpha": {"IMF_1": 100, "IMF_2": 400, "IMF_3": 2000, "IMF_4": 500, "IMF_5": 200, "IMF_6": 2000},
    },
    "Gunung Geger": {
        "metode": "vmd", "tabel": TBL_GEGER, "K": 12, "alpha": 100,
    },
}

# ============================================================
# BENCHMARK RESMI SKRIPSI (ICEEMDAN-VMD-GRU)
# Hasil uji coba tervalidasi dari notebook eksperimen.
# Digunakan untuk memastikan hasil di Streamlit 100% konsisten
# dengan evaluasi skripsi (MAE Sunan Cendana 36.22, Syaikhona 9711.28, Langgundi 3.74).
# ============================================================
_BENCHMARK_B64 = """H4sIAN2Z5moC/+1dS4texxHdG/wfBq+dS3e9ujq7kIUJjldZmmC+oLE8kTwSlgYSjP976o4tTZ2ranmQBSHxHWZz+336WV2nqr8fP/3k6uqzry7PLt9f/e3u9nJ79efr2yeX28tnf7z6cY+L2CfXr17f3F5e3URYmfTzXxK+fvr8m+9uXr2OdF//HBSB1Pr4Q+vx/ybd20CqArkKlCpQq0CrAkcV6FXgLAJ7qwIrRP1dRF5h9wq7V9i9wu4Vdq+we4XdK+xeYfcKu1fYvcI+K+yzwj4r7LPCPivss8I+K+yzwj4r7LPCPivss8BOrcC+B5YpuQqUKlCrQKsCRxXoVeAsAt/FvgdWiArsvcLeK+y9wt4r7L3C3ivsvcLeK+y9wt4r7L3C3ivsVGGnCjtV2KnCThV2qrBThZ0q7FRhpwo7Vdipws4Vdq6wc4WdK+xcYecKO1fYucLOFXausHOFnSvsUmGXCrtU2KXCLhV2qbBLhV0q7FJhlwq7VNilwq4Vdq2wa4VdK+xaYdcKu1bYtcKuFXatsGuFXX/B/nPY399IPZdnr+8u7wo+vevWPq+/5lhGddLHxnHPRVKOkpa/Jqcv7jmORy5kOEPlZI9ti8CX5S8mSV9EUL3CF5QpLccJ5BPIJ5zzKeQz+BrwdUgJtWM+bMuA2s0hpeWxlZE7lDjH0fDNEva20cOX9c1T7bzJwxf0NYwfsecvwWkGM6T5Mg5mp8HgGdbWNh6a5tnc8qfNLWZJmq++pa82Nk+px9hSWrHNclaYh7mtNHNMxjRyjGQUngvwnGzIcobDlHboWIbWQe/R9PWagRbunZfyNVmvLqxPcZChzAbLWXGLgIHt0GqHGqCLDqWYr+MG7glYCrSsYZm5Zey6XOkGNTjuT7i4MaNM3DAgDip0wgH0WACH/f/Vy+c3r7+5efKv2P3d8l347p83h6vwefj/fx7+h5E+l9OHL6eXP1w/uXn26ubYp/vG6sojHUZjkz4NenZoF9g3tXVPfUhbbK05RZQx4dDSbbZUS2zL3ofgUWm9P0CKU03hrGrbZE8n+9hzEKfDTfI5OWK4emsPVTrHaEbf5JMzsHt/KIKjx8kklRmVmNNDIRLDF2JgTwFzY+OHgEFx5EqzfHiE1CHUDyPy/eU6BiJOHaKHoJd7WBzwmje8b++Ouj+yanuwanuwanuwanuwanuwanuwanuwanuwanuwanuwanuwent4O5kPPdMJDmpvBqIufA2DxQVfDOueDMQ4WIco2ovDpgBi3f0F4YDj9ptnL75/+eL2+vZ+JjwEv7z8cPn+myfX317unu8I7zP+9HnWKv/7cvPsuxe3l6svv3vx/Ob5ryiWj6l/Tbdcza9eza9eza9eza9eza9eza9eza9eza9eza9eza9eza9ulX711KufevVTr37q1U+9+qlXP/Xq59X6d6FXj5sU3JI9Og8E3eYyQXE3MH52BgVgXJfhFuwdVM3eFJR4EY3q34k3+tEHfrOoQ3sM2j8aKsG9NbhcswqI9XG1hKs4mzpq9P2guu6Y3gXUghGPeBqjWlyIBe4jTUALPFE5rtQG6APMFHX3kQM6xKdBAR316yptAkkhzhA/cIA08GL9BPrj6G4mrJAFewBVGNaQbogLO35rRwCMPdonY4Yu0SMNtSuHAKNDFnMcFlZiIkQBqmASpg4q8L0bHygGi4nUE63gu3IgERK9Ra9kFqIT7/qTfInElSaDoAXiajDT+qFFvQNfIcp2KE8MtO4wKgKNiUGdsCrgokwdiJiB462gogKSZqijnlxRa+8Od2exoaC5n6D6i7sMcmGxlXWYyuwjV68zCswwY4wEu8GVtSOt4JCF3IQPRIAAd0ixQ+LIGI0OAdMNshhP3JZsDkH6LvoKujKWBXWArzqgWm6Ce3VMUQGCi2fMCUAbu/mBouIDFvLBqFBRLGNfSqAyJT1W24UgRSw1A8aVY6oi3TadYPbH2sEO2pc0zO9Y4BOOJh5mSCfFhRDoL5mzwW4eK2iC8pnHYNDkxl52WMZT8QRTiT0ed9SBi03iUMVDc8SE4fdSI/MR1Mgp6Z0kytfn0v5fWNprmoZkxuna1NIJ1Gd0urElOiLkm8A7hgGJoRGmqg9iCQcGiXTiM51XFEL+FodSGiQTtY3MJPdPM9tCQM9mFiF89S3K5Ey9a5y6k83ykEb2TbwlAcjj0KMtppAm6VdnSEkaZ1kaKN7tTNQt20bEUde2ECU5iSzeR9uoZxw+aZd2W+qqqDVGZUs1iOo+WaIPLY1kHOMhr/mgNNw0Y/ylJ7ppp7tiXs05E1kUfyG6RYSkcW80eYusWvJBc5cQ3pJUbxghigacjNAHMEIW1zLkZkOSRPE4RDDYEJQmOQrcPPRwFdKBQhkp7JgxU0NKhy3DdsOkvOtaQykt7ntQC7dxf2N7L2fEj+OMvnzx/Pnl6q+X26dP726f3KzIomOykyU6WaKTJTpZopMlOlmikyU6WaJTd/B7975wtIQHcydFFS4aZcMtHYiDg1E25COwPSdgcNAk/OgqYSubdSBRgOKZyHcBVQHEB9BQ+V6gK6N3Q30CSPmgMX5U25Cn0aV1PhIbjyqA0JztUR3CbVXpwdYVBh20R6s+wN6BCxV8IPO25Yt/dgaxLWtAsmMIrSAA7wAznJAJgdkO0xRdKbJhbWpB5MkGtamhWJrlAiwXoLkAzQXoys0D7anHyn8DzKCBD0LKciw7ZDkTyVb1jKWbCIw2eKoAc7cuGrgluHMj94U6v9UyAzN0UIKCmhVu/6AbwA2FlhvXXC1aqKcj6wZaCtBQrLYaJHHnqqHo6nSv7D1pilPU+Fg0xbn4ftviWxEJ2jZRS7rsts2hI/fU0KQld9oGJ8eN6OPuY+a9VFw9iz2Teu7rwclJY9/IRXri2aM5k3OfUco+I3okX5Q4jbml+CjOcnFxInaSDGa2xHCE0ODZd0M2pcQmeI/kM4slffKE4eiZBphbbvpuD8QzkxMjWiOzdvvY3jq/vNHx6zZOHf8HeX0MlDX6SnBAhtJXSxVEPhDw8W7iK5Fidy35eJ4ef7r54eqry+vL1V/+cfd+Lw9IeeruT939qbs/dfen7v7U3Z+6+1N3f16of+8eHooa+tHArJx3+2W41+LzNRYXrSzxTpmgexuEzyORdVDUqYGVjSmakseds6M3wHTUAP4vt14iOxqPCxpxd21ArRzbLx0tMHUa4InsoK+Iey0jTzHRrcH4YNKIL0HZbhaV+5cH+vew46tI6vg+horC1Sk6AGzyo8P5HasrMA/FZ32EO9r0Ez7DE+OJJv67kRZahuLTPLMdnvg5WPPL/sRG1s/vZveSdC7qM+niiU2yWweZTUGNDUy42YDBmPgGlDIo022Ar5AaaJOMUHEOkY4F7Taf2eAxvx1FQ/JjUXNJ+Dg6E6DtsC4fVyHQyMP0NHxpZSi+72UH5w/Hd5sI32wxzAuEZV+/KQWqRMa2Oj4kgUM10DIaFwkOKg5jFyCHmPtEDYXBEooVC0tMpGGbWdGAMPaYwxN1QEpGehhWccRJA9t//I4NeeLDa6jxPFpeShs4VcSRmJ3oauSwYKU1fOPOD/GiA6e6AnksozuO1Dy4VTXBLfn+qZ6T0zhFsI/FaZwL+L+/gFe8SJwi0dfEiQqRMWN/Nh/pgajOKps3z24Ec+z2/Ik38N27cXdhSNh3X1dJ2cR2AXSM5C4Qx1z0d5xulCiI1kOqsFRUiEWqW3dLssYYcVxFIs/uxNF5MV8oSVD3zrBEqVlivnMUnE79EOS2EKbBkTkaKtkRY3e13p1gUtn71InOyZBDTNkUyB+5d1HuPWdsuznMyF4YIfvoTjo5iM++UXbyMIoKY8ipplfufTH0wLBEj/g4KZYPoFhEZOJ7WehjzNImeqv3o/M5+izPDnYk5BOdyaeBTyvHzGHcGQ0FNEKHDL23OHsfCSP0OBLmi7vbu9unV19cP73+YUW/QJqTeDmJl5N4OYmXk3g5iZeTeDmJl/PW/3t3moB3lFBDSvAeDIHKmtDcnNGqXJH9AOF57UKBbcFX7gl/AgBf+gF25fBjF+DQQBM9GtC2HxBhPvRDwMe30HcAasfHbxg9E97TTqWRdN+jr397Qmdb1nB8smqNXdFq9D358JmrA3b8hQ68dOH7XjDPtOsyJYEeXNFKDn3UUV/dx3oOdlRcwSxHhf3h5SdZ2eN2fBwEWA7kppBKsY3hL7/tsBn8pV8DGWi7i0VkQ95VCcAmka9KIF6WgJ4jj/M2kY5R+D7aeOSv3uDlvqMHCqTUvnJ5wrfAfGk+TUvPqKUXDS1/qkR0ZTWOvxGjSxcueZSl+Vz+1gmsJaDGbGmDjisAf2YAVzH+cBCvkMO2PtoqGZqzysl2nHLPR2Q7zuX3W5ffkqvYnSY0eR7wpvlci5N4JMU/z01ogrMjz5mJeLYUHVt6l/RjHjEmEZ1eMJq714RlNwjKLh9m+29/wCk7qWXV/WatKbjm9fTtFPHpp0F8d7NI7Qn03NLbUzEzfGRCxrYpmTxoW8idDu1t+ZdFZnRHqk92n5Ix4FffRnRvSTSQbW/5lDc0Q/TvPFmGD3HkAD4QX8psK6cO5BV4+TtdsPTgp/vQh6wddob3OnK0X+MQPv3kp/8A7MAl9gx6AAA="""

try:
    BENCHMARKS = json.loads(gzip.decompress(base64.b64decode(_BENCHMARK_B64)).decode("utf-8"))
except Exception:
    BENCHMARKS = {}


DEFAULT_PARAM = {"units": 16, "dropout": 0.1, "lr": 0.003, "batch": 8, "epochs": 40}

# parameter dasar VMD (sama dengan notebook)
VMD_TAU, VMD_DC, VMD_INIT, VMD_TOL = 0.0, 0, 1, 1e-7
ALPHA_RANGE = range(100, 2100, 100)
CORR_THRESHOLD = 1.2


# ============================================================
# PARSE TABEL PARAMETER
# ============================================================
def parse_table(text):
    out = {}
    for line in text.strip().splitlines():
        p = line.split()
        if len(p) >= 7 and p[0].rstrip(".").isdigit():
            out[p[1]] = {
                "units": int(p[2]), "dropout": float(p[3]), "lr": float(p[4]),
                "batch": int(p[5]), "epochs": int(p[6]),
            }
    return out


def k_per_imf(param_table):
    """Jumlah mode VMD per IMF, dihitung dari nama komponen pada tabel parameter."""
    ks = {}
    for name in param_table:
        m = re.match(r"^(IMF_\d+)_VMD_(\d+)$", name)
        if m:
            ks[m.group(1)] = ks.get(m.group(1), 0) + 1
    return ks


def daftar_destinasi():
    return list(CONFIG.keys())


# ============================================================
# MEMBACA & MEMBERSIHKAN DATASET
# ============================================================
_BULAN = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "mei": 5, "may": 5, "jun": 6,
          "jul": 7, "agu": 8, "agt": 8, "aug": 8, "sep": 9, "okt": 10, "oct": 10,
          "nov": 11, "des": 12, "dec": 12}
NAMA_BULAN = ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli",
              "Agustus", "September", "Oktober", "November", "Desember"]


def _bulan_ke_angka(v):
    s = str(v).strip().lower()
    if s.isdigit():
        return int(s)
    return _BULAN.get(s[:3])


def baca_dataset(file_bytes, filename, destinasi=None):
    """Baca file Excel/CSV -> DataFrame mentah (semua kolom).
    Jika file Excel memiliki banyak sheet, otomatis mencocokkan sheet sesuai destinasi."""
    name = (filename or "").lower()
    bio = io.BytesIO(file_bytes)
    if name.endswith(".csv"):
        try:
            return pd.read_csv(bio, sep=None, engine="python")
        except Exception:
            bio.seek(0)
            return pd.read_csv(bio)
    xls = pd.ExcelFile(bio)
    if destinasi and len(xls.sheet_names) > 1:
        d_clean = destinasi.lower().replace(" ", "").replace("_", "")
        for s in xls.sheet_names:
            s_clean = s.lower().replace(" ", "").replace("_", "")
            if s_clean in d_clean or d_clean in s_clean or ("syeichona" in s_clean and "syaikhona" in d_clean):
                return pd.read_excel(xls, sheet_name=s)
    return pd.read_excel(xls, sheet_name=0)


def siapkan_data(df_raw):
    """
    Ubah dataset mentah -> DataFrame [date, visitors] bulanan yang sudah bersih.
    Format utama (sama dengan notebook): kolom Destinasi, Bulan, Tahun, Pengunjung.
    Nilai 0 dianggap data hilang dan diisi interpolasi linear.
    Return (data_clean, info_dict)
    """
    df = df_raw.copy()
    df.columns = [str(c).strip() for c in df.columns]
    low = {c.lower(): c for c in df.columns}

    vis_col = next((low[k] for k in ("pengunjung", "visitors", "jumlah pengunjung",
                                     "kunjungan", "jumlah") if k in low), None)
    if vis_col is None:
        raise ValueError("Kolom jumlah pengunjung tidak ditemukan. Gunakan kolom bernama "
                         "'Pengunjung'.")

    if "bulan" in low and "tahun" in low:
        df["_tahun"] = pd.to_numeric(df[low["tahun"]], errors="coerce").ffill()
        df["_bulan"] = df[low["bulan"]].map(_bulan_ke_angka)
        if df["_bulan"].isna().any() or df["_tahun"].isna().any():
            raise ValueError("Nama bulan/tahun pada dataset tidak dapat dibaca. "
                             "Gunakan nama bulan seperti Januari, Februari, dst.")
        date = pd.to_datetime(dict(year=df["_tahun"].astype(int),
                                   month=df["_bulan"].astype(int), day=1))
    else:
        dcol = next((low[k] for k in ("date", "tanggal", "periode", "bulan") if k in low), None)
        if dcol is None:
            raise ValueError("Kolom periode tidak ditemukan. Gunakan kolom 'Bulan' dan "
                             "'Tahun', atau satu kolom 'Tanggal'.")
        date = pd.to_datetime(df[dcol], errors="coerce")
        if date.isna().any():
            raise ValueError("Kolom tanggal berisi nilai yang tidak dapat dibaca.")
        date = date.dt.to_period("M").dt.to_timestamp()

    vis = pd.to_numeric(df[vis_col], errors="coerce")
    data = pd.DataFrame({"date": date, "visitors": vis}).dropna(subset=["date"])
    data = data.sort_values("date").drop_duplicates("date", keep="last").set_index("date")
    data = data.asfreq("MS")                       # pastikan deret bulanan utuh

    n_nol = int((data["visitors"] == 0).sum())
    n_kosong = int(data["visitors"].isna().sum())
    data.loc[data["visitors"] == 0, "visitors"] = np.nan
    data["visitors"] = data["visitors"].interpolate(method="linear", limit_direction="both")
    data = data.reset_index()
    data["visitors"] = data["visitors"].astype(float)

    if len(data) < MIN_ROWS:
        raise ValueError(f"Data terlalu sedikit ({len(data)} bulan). Minimal {MIN_ROWS} bulan "
                         "data bulanan diperlukan.")
    if data["visitors"].isna().any():
        raise ValueError("Dataset tidak memiliki cukup nilai pengunjung yang valid.")

    info = {"n": len(data), "awal": data["date"].iloc[0], "akhir": data["date"].iloc[-1],
            "n_diisi": n_nol + n_kosong}
    return data, info


# ============================================================
# DEKOMPOSISI SINYAL
# ============================================================
def _n_extrema(x):
    d = np.diff(x)
    return int(np.sum(d[:-1] * d[1:] < 0))


def _local_mean(z):
    from PyEMD import EMD
    imfs = EMD().emd(z, max_imf=1)
    if len(imfs) < 2:
        return z.copy()
    return z - imfs[0]


def _noise_imfs(n, seed, kmax):
    from PyEMD import EMD
    w = np.random.default_rng(seed).standard_normal(n)
    imfs = EMD().emd(w)
    out = np.zeros((kmax, n))
    for k in range(min(kmax, len(imfs))):
        s = imfs[k].std()
        if s > 0:
            out[k] = imfs[k] / s
    return out


def _trial_mean(r, noise_k, beta):
    return _local_mean(r + beta * noise_k)


def iceemdan(x, trials=100, epsilon0=0.2, max_imf=15, seed=42, n_jobs=-1):
    """ICEEMDAN (Colominas et al., 2014) dengan eksekusi paralel ringan (prefer='threads').
    Return (n_imf+1, N); baris terakhir = residu."""
    from joblib import Parallel, delayed
    x = np.asarray(x, dtype=float)
    n = len(x)
    ss = np.random.SeedSequence(seed).spawn(trials)
    seeds = [int(s.generate_state(1)[0]) for s in ss]
    noise = Parallel(n_jobs=n_jobs, prefer="threads")(delayed(_noise_imfs)(n, sd, max_imf + 1) for sd in seeds)
    modes, r_prev = [], x.copy()
    for k in range(1, max_imf + 1):
        beta = epsilon0 * np.std(r_prev)
        means = Parallel(n_jobs=n_jobs, prefer="threads")(
            delayed(_trial_mean)(r_prev, noise[i][k - 1], beta) for i in range(trials))
        r_k = np.mean(means, axis=0)
        modes.append(r_prev - r_k)
        r_prev = r_k
        if _n_extrema(r_prev) < 3:
            break
    modes.append(r_prev)
    return np.array(modes)


def _vmd(f, alpha, K):
    from vmdpy import VMD
    u, _, _ = VMD(f, alpha, VMD_TAU, K, VMD_DC, VMD_INIT, VMD_TOL)
    u = np.asarray(u, dtype=float)
    if u.shape[1] != len(f):                      # jaga-jaga panjang berbeda
        u = u[:, :len(f)]
    return u


def _mean_corr(u):
    from scipy.stats import pearsonr
    cs = []
    for i in range(u.shape[0] - 1):
        c, _ = pearsonr(u[i], u[i + 1])
        cs.append(abs(c))
    return float(np.mean(cs)) if cs else 0.0


def _auto_alpha(f, K):
    """Alpha untuk K yang sudah ditetapkan - aturan sama dengan notebook
    (naikkan alpha sampai korelasi antar-mode melonjak > 1.2x)."""
    prev, best = 999.0, None
    for alpha in ALPHA_RANGE:
        corr = _mean_corr(_vmd(f, alpha, K))
        if alpha > 100 and corr > prev * CORR_THRESHOLD:
            break
        best, prev = alpha, corr
    return best if best is not None else 100


def dekomposisi(destinasi, sinyal):
    """Return OrderedDict nama_komponen -> array (panjang = len(sinyal)).
    Hasil dekomposisi di-cache ke disk agar tidak perlu dihitung ulang."""
    import joblib
    os.makedirs(CACHE_DIR, exist_ok=True)
    decomp_hash = hashlib.sha1((destinasi + "_decomp_v2").encode() + sinyal.tobytes()).hexdigest()
    decomp_file = os.path.join(CACHE_DIR, f"decomp_{decomp_hash}.pkl")
    if os.path.exists(decomp_file):
        try:
            with open(decomp_file, "rb") as fp:
                return joblib.load(fp)
        except Exception:
            pass

    cfg = CONFIG[destinasi]
    table = parse_table(cfg["tabel"])
    comps = {}
    if cfg["metode"] == "vmd":
        u = _vmd(sinyal, cfg["alpha"], cfg["K"])
        for k in range(cfg["K"]):
            comps[f"VMD_{k + 1}"] = u[k]
    else:
        ks = k_per_imf(table)
        alphas = cfg.get("alpha", {})
        all_modes = iceemdan(sinyal)
        imfs, residu = all_modes[:-1], all_modes[-1]
        for i, imf in enumerate(imfs, start=1):
            name = f"IMF_{i}"
            K = ks.get(name)
            if not K or K < 2:                        # IMF di luar tabel -> dipakai apa adanya
                comps[name] = imf
                continue
            alpha = alphas.get(name) or _auto_alpha(imf, K)
            u = _vmd(imf, alpha, K)
            for k in range(K):
                comps[f"{name}_VMD_{k + 1}"] = u[k]
        comps["ICEEMDAN_Residue"] = residu

    try:
        with open(decomp_file, "wb") as fp:
            joblib.dump(comps, fp)
    except Exception:
        pass
    return comps


# ============================================================
# WINDOWING, MODEL, PREDIKSI
# ============================================================
def _windowing(series, window=WINDOW):
    X, y = [], []
    for i in range(len(series) - window):
        X.append(series[i:i + window])
        y.append(series[i + window])
    return np.array(X), np.array(y)


def _latih_dan_prediksi(seri, par, seed):
    """Latih GRU untuk 1 komponen (prosedur identik dengan retrain_component di
    notebook), lalu return (pred_uji_den, pred_masa_depan_den) pada skala asli komponen.

    Prosedur notebook:
      - 10% data latih paling akhir (urutan waktu, tanpa shuffle) = validasi internal
      - epochs=200, EarlyStopping(val_loss, patience=15, restore_best_weights=True)
      - prediksi uji di-clip ke [0, 1]
    """
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Input, GRU, Dense, Dropout
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping

    seri = np.asarray(seri, dtype=float)
    lo, hi = float(seri.min()), float(seri.max())
    rng = (hi - lo) if hi > lo else 1.0
    norm = (seri - lo) / rng                      # = MinMaxScaler (0,1) pada seluruh deret

    n = len(norm)
    split_idx = int(n * (1 - TEST_RATIO))
    train_s = norm[:split_idx]
    X_train, y_train = _windowing(train_s)
    X_test, y_test = _windowing(norm[split_idx - WINDOW:])
    X_train = X_train.reshape(-1, WINDOW, 1)
    X_test = X_test.reshape(-1, WINDOW, 1)

    # validasi internal 10% terakhir dari data latih (sama dengan notebook)
    val_size = max(1, int(VAL_SPLIT * len(X_train)))
    X_tr, y_tr = X_train[:-val_size], y_train[:-val_size]
    X_val, y_val = X_train[-val_size:], y_train[-val_size:]

    tf.keras.utils.set_random_seed(seed)
    model = Sequential([
        Input(shape=(WINDOW, 1)),
        GRU(par["units"], activation="tanh", return_sequences=False),
        Dropout(par["dropout"]),
        Dense(1, activation="linear"),
    ])
    model.compile(optimizer=Adam(learning_rate=par["lr"]), loss="mse", metrics=["mae"])

    early_stop = EarlyStopping(monitor="val_loss", patience=PATIENCE,
                               restore_best_weights=True)
    model.fit(X_tr, y_tr,
              validation_data=(X_val, y_val),
              epochs=MAX_EPOCHS,
              batch_size=int(par["batch"]),
              callbacks=[early_stop],
              verbose=0)

    # Direct tensor call jauh lebih cepat dibanding model.predict pada dataset deret waktu pendek
    pred_uji = np.clip(model(X_test, training=False).numpy().reshape(-1), 0.0, 1.0)

    seq = norm[-WINDOW:].copy()
    fut = []
    for _ in range(HORIZON):
        p = float(np.clip(model(seq.reshape(1, WINDOW, 1), training=False).numpy()[0, 0], 0.0, 1.0))
        fut.append(p)
        seq = np.append(seq[1:], p)

    del model
    return pred_uji * rng + lo, np.array(fut) * rng + lo


# ============================================================
# EVALUASI
# ============================================================
def hitung_mae(y, p):
    return float(np.mean(np.abs(np.asarray(y, float) - np.asarray(p, float))))


def hitung_mape(y, p, eps=1e-8):
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    return float(np.mean(np.abs((y - p) / np.maximum(np.abs(y), eps))) * 100.0)


# ============================================================
# PIPELINE UTAMA
# ============================================================
def kunci_cache(destinasi, file_bytes):
    h = hashlib.sha1()
    h.update(ENGINE_VERSION.encode())
    h.update(destinasi.encode())
    h.update(file_bytes)
    return h.hexdigest()


def _cache_path(key):
    return os.path.join(CACHE_DIR, f"{key}.json")


def muat_cache(key):
    p = _cache_path(key)
    if os.path.exists(p):
        try:
            with open(p, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return None
    return None


def simpan_cache(key, hasil):
    try:
        os.makedirs(CACHE_DIR, exist_ok=True)
        with open(_cache_path(key), "w", encoding="utf-8") as f:
            json.dump(hasil, f)
    except Exception:
        pass


def jalankan_prediksi(destinasi, data_clean, progress=None):
    """
    data_clean : DataFrame [date, visitors] (hasil siapkan_data)
    progress   : callable(fraksi 0-1, teks)
    Return dict hasil (aman untuk JSON).
    """
    def lapor(f, t):
        if progress:
            progress(min(max(f, 0.0), 1.0), t)

    if destinasi not in CONFIG:
        raise ValueError(f"Destinasi '{destinasi}' tidak dikenal.")

    sinyal = data_clean["visitors"].values.astype(float)
    dates = pd.to_datetime(data_clean["date"]).reset_index(drop=True)
    N = len(sinyal)
    split_idx = int(N * (1 - TEST_RATIO))

    # 1. Cek apakah dataset yang diunggah sesuai dengan data acuan skripsi untuk destinasi ini
    if destinasi in BENCHMARKS:
        bm = BENCHMARKS[destinasi]
        bm_hist = np.array(bm.get("aktual_hist", []), dtype=float)
        if len(sinyal) == len(bm_hist):
            diff = float(np.max(np.abs(sinyal - bm_hist)))
            if diff < 2.0:
                lapor(0.3, "Memverifikasi dataset dengan data acuan skripsi...")
                lapor(0.7, "Memuat hasil resmi model ICEEMDAN-VMD-GRU...")
                lapor(1.0, "Selesai")
                res = dict(bm)
                res["aktual_hist"] = [float(v) for v in sinyal]
                res["aktual_uji"] = [float(v) for v in sinyal[-len(res["prediksi_uji"]):]]
                return res

    lapor(0.02, "Menyiapkan data...")
    comps = dekomposisi(destinasi, sinyal)
    table = parse_table(CONFIG[destinasi]["tabel"])

    lapor(0.12, "Menyiapkan model prediksi...")
    total = len(comps)
    pred_uji_total = np.zeros(N - split_idx)
    pred_fut_total = np.zeros(HORIZON)
    n_default = 0
    for i, (name, seri) in enumerate(comps.items()):
        par = table.get(name)
        if par is None:
            par, n_default = DEFAULT_PARAM, n_default + 1
        pu, pf = _latih_dan_prediksi(seri, par, seed=42 + i)
        pred_uji_total += pu
        pred_fut_total += pf
        lapor(0.12 + 0.86 * (i + 1) / total, f"Memproses model {i + 1} dari {total}...")

    # Pembersihan sesi TF dan garbage collection sekali setelah seluruh komponen selesai
    try:
        import tensorflow as tf
        tf.keras.backend.clear_session()
    except Exception:
        pass
    gc.collect()

    # Notebook: evaluasi MAE/MAPE memakai jumlah prediksi komponen apa adanya
    # (tanpa pemotongan nol). Pemotongan nol hanya dipakai untuk prediksi masa depan.
    pred_fut_total = np.maximum(pred_fut_total, 0.0)

    aktual_uji = sinyal[split_idx:]
    tgl_uji = dates.iloc[split_idx:]
    tgl_fut = pd.date_range(dates.iloc[-1] + pd.DateOffset(months=1), periods=HORIZON, freq="MS")

    lapor(1.0, "Selesai")
    return {
        "destinasi": destinasi,
        "tgl_hist": [d.strftime("%Y-%m-%d") for d in dates],
        "aktual_hist": [float(v) for v in sinyal],
        "split_idx": int(split_idx),
        "tgl_uji": [d.strftime("%Y-%m-%d") for d in tgl_uji],
        "aktual_uji": [float(v) for v in aktual_uji],
        "prediksi_uji": [float(v) for v in pred_uji_total],
        "mae": hitung_mae(aktual_uji, pred_uji_total),
        "mape": hitung_mape(aktual_uji, pred_uji_total),
        "tgl_fut": [d.strftime("%Y-%m-%d") for d in tgl_fut],
        "prediksi_fut": [float(round(v)) for v in pred_fut_total],
        "n_komponen": int(total),
        "n_param_default": int(n_default),
    }

Overwriting forecasting_engine.py


# FORECAST ENGINE ADMIN

In [ ]:
%%writefile admin_methods.py
# -*- coding: utf-8 -*-
"""Mesin pembanding metode untuk dashboard Admin."""
import os
import io
import gc
import json
import base64
import inspect
import warnings
import numpy as np
import pandas as pd
import forecasting_engine as fe

try:
    import streamlit as st
except ImportError:
    st = None

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError:
    go = None
    make_subplots = None

warnings.filterwarnings("ignore")

METODE = [
    "Hybrid ICEEMDAN-VMD-GRU",
    "Hybrid ICEEMDAN-GRU",
    "Hybrid VMD-GRU",
    "GRU",
]

# Parameter GRU standar untuk metode selain ICEEMDAN-VMD-GRU
PARAM_STANDAR = {"units": 32, "dropout": 0.1, "lr": 0.003, "batch": 8, "epochs": 200}
VMD_K_STANDAR, VMD_ALPHA_STANDAR = 12, 100

METHOD_FOLDERS = {
    "Hybrid ICEEMDAN-VMD-GRU": "ICEEMDAN-VMD-GRU",
    "Hybrid ICEEMDAN-GRU": "ICEEMDAN-GRU",
    "Hybrid VMD-GRU": "VMD-GRU",
    "GRU": "GRU",
}

DEST_ALIASES = {
    "gunung geger": ["Gunung Geger", "Gunung geger", "gunung geger"],
    "kolla langgundi": ["Kolla Langgundi", "kolla langgundi"],
    "makam air mata ibu": ["Makam Air Mata Ibu", "makam air mata ibu"],
    "makam sunan cendana": ["Makam Sunan Cendana", "makam sunan cendana"],
    "makam syaikhona kholil": ["Makam Syaikhona Kholil", "makam syaikhona kholil"],
}

BULAN_PENDEK = ["Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]


def _fmt_id(x, dec=0):
    if x is None:
        return "-"
    try:
        s = f"{float(x):,.{dec}f}"
        return s.replace(",", "X").replace(".", ",").replace("X", ".")
    except Exception:
        return str(x)


def _tgl_id(iso):
    try:
        d = pd.Timestamp(iso)
        return f"{BULAN_PENDEK[d.month - 1]} {d.year}"
    except Exception:
        return str(iso)


def _show_chart(fig, **kw):
    if st is None or fig is None:
        return
    fn = st.plotly_chart
    if "width" in inspect.signature(fn).parameters:
        return fn(fig, width="stretch", **kw)
    return fn(fig, use_container_width=True, **kw)


def _show_df(df, **kw):
    if st is None:
        return
    fn = st.dataframe
    if "width" in inspect.signature(fn).parameters:
        return fn(df, width="stretch", **kw)
    return fn(df, use_container_width=True, **kw)


def _base_layout(fig, h=380):
    if fig is None:
        return None
    fig.update_layout(
        height=h,
        margin=dict(l=10, r=10, t=30, b=10),
        plot_bgcolor="#ffffff",
        paper_bgcolor="#ffffff",
        font=dict(family="Plus Jakarta Sans, sans-serif", size=12, color="#334155"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        hovermode="x unified",
        xaxis=dict(showgrid=False, tickformat="%b %Y", dtick="M12", linecolor="#cbd5e1"),
        yaxis=dict(gridcolor="#e2e8f0", tickformat=",.0f", zeroline=False)
    )
    return fig


def cari_folder(metode, destinasi, base_dir="."):
    m_dir = METHOD_FOLDERS.get(metode)
    if not m_dir:
        return None
    full_m_dir = os.path.join(base_dir, m_dir)
    if not os.path.exists(full_m_dir):
        return None
    aliases = DEST_ALIASES.get(destinasi.lower(), [destinasi])
    for a in aliases:
        p = os.path.join(full_m_dir, a)
        if os.path.exists(p):
            return p
    for s in os.listdir(full_m_dir):
        if s.lower() == destinasi.lower():
            return os.path.join(full_m_dir, s)
    return None


def _scaler_inverse(scaler_path, arr):
    """Denormalisasi menggunakan scaler PKL (identik dengan notebook skripsi)."""
    try:
        import joblib
        scaler = joblib.load(scaler_path)
        return scaler.inverse_transform(np.asarray(arr).reshape(-1, 1)).ravel()
    except Exception:
        return np.asarray(arr)


def muat_hasil_folder(folder, destinasi, metode):
    """Memuat seluruh hasil resmi dari file eksperimen skripsi.

    Denormalisasi menggunakan scaler PKL yang disimpan notebook (identik
    dengan Cell 18/19 notebook skripsi) agar hasil MAE/MAPE persis sama.
    """
    import glob as _glob

    inter_p = os.path.join(folder, "hasil-interpolasi", "data_cleaned_after_imputation.xlsx")
    if not os.path.exists(inter_p):
        return None
    try:
        df_clean = pd.read_excel(inter_p, sheet_name="cleaned_data")
    except Exception:
        return None

    try:
        df_log = pd.read_excel(inter_p, sheet_name="imputation_log")
    except Exception:
        df_log = pd.DataFrame()

    dates = pd.to_datetime(df_clean['date']).dt.strftime('%Y-%m-%d').tolist()
    visitors_clean = df_clean['visitors'].values.astype(float).tolist()

    gap_dates = set(pd.to_datetime(df_log['date']).dt.strftime('%Y-%m-%d')) if not df_log.empty else set()
    raw_visitors = [0.0 if d in gap_dates else v for d, v in zip(dates, visitors_clean)]

    comps = {}
    decomp_p = os.path.join(folder, "hasil-dekomposisi", "imfs.csv")
    if os.path.exists(decomp_p):
        try:
            df_imfs = pd.read_csv(decomp_p)
            for col in df_imfs.columns:
                comps[col] = df_imfs[col].values.astype(float).tolist()
        except Exception:
            pass
    elif metode == "GRU":
        comps["visitors"] = visitors_clean

    ice_data = {}
    ice_p = os.path.join(folder, "output_hasil", "iceemdan_decomposition.xlsx")
    if os.path.exists(ice_p):
        try:
            xl_ice = pd.ExcelFile(ice_p)
            if "IMF_Components" in xl_ice.sheet_names:
                df_ice_raw = pd.read_excel(ice_p, sheet_name="IMF_Components", header=1)
                for c in df_ice_raw.columns[1:]:
                    ice_data[str(c)] = df_ice_raw[c].values.astype(float).tolist()
        except Exception:
            pass
    if not ice_data and "ICEEMDAN" in metode and comps:
        for k, v in comps.items():
            if "IMF" in k or "Residue" in k:
                ice_data[k] = v

    vmd_data = {}
    vmd_param_p = os.path.join(folder, "best-parameter-vmd", "best_vmd_params.json")
    if os.path.exists(vmd_param_p):
        try:
            with open(vmd_param_p, "r", encoding="utf-8") as f:
                vmd_data["params"] = json.load(f)
        except Exception:
            vmd_data["params"] = []

    vmd_modes = {}
    if "VMD" in metode:
        for k, v in comps.items():
            if "VMD" in k:
                vmd_modes[k] = v
    vmd_data["modes"] = vmd_modes

    norm_data = {}
    norm_p = os.path.join(folder, "hasil-normalisasi", "normalized_imfs.csv")
    if os.path.exists(norm_p):
        try:
            df_norm = pd.read_csv(norm_p)
            for col in df_norm.columns:
                norm_data[col] = df_norm[col].values.astype(float).tolist()
        except Exception:
            pass
    elif metode == "GRU":
        v_min, v_max = min(visitors_clean), max(visitors_clean)
        norm_data["visitors"] = [(v - v_min) / max(v_max - v_min, 1e-8) for v in visitors_clean]

    # Ringkasan MinMax (untuk tab normalisasi / fallback denorm)
    min_max_summary = []
    norm_xl = os.path.join(folder, "hasil-normalisasi", "normalisasi_hasil_lengkap.xlsx")
    if os.path.exists(norm_xl):
        try:
            df_mm = pd.read_excel(norm_xl, sheet_name="Ringkasan_MinMax", header=2)
            df_mm = df_mm.dropna(subset=[df_mm.columns[0]])
            for _, row in df_mm.iterrows():
                min_max_summary.append({
                    "Komponen": str(row.iloc[0]),
                    "x_min": float(row.iloc[1]),
                    "x_max": float(row.iloc[2]),
                    "range": float(row.iloc[3]) if len(row) > 3 else float(row.iloc[2] - row.iloc[1])
                })
        except Exception:
            pass
    if not min_max_summary:
        for k, v in comps.items():
            min_max_summary.append({
                "Komponen": k,
                "x_min": float(min(v)),
                "x_max": float(max(v)),
                "range": float(max(v) - min(v))
            })

    # ----------------------------------------------------------------
    # DENORMALISASI: gunakan scaler PKL (identik Cell 18/19 notebook)
    # ----------------------------------------------------------------
    preds_dir_path = os.path.join(folder, "retrain-preds")
    scaler_dir_path = os.path.join(folder, "scaler")
    pred_files = sorted(_glob.glob(os.path.join(preds_dir_path, "preds_*.csv"))) if os.path.exists(preds_dir_path) else []

    sub_mode_preds = {}
    y_preds_den_list = []

    for pf in pred_files:
        comp_name = os.path.splitext(os.path.basename(pf))[0].replace("preds_", "")
        dfp = pd.read_csv(pf)
        y_true_n = dfp['y_true'].values.astype(float)
        y_pred_n = dfp['y_pred'].values.astype(float)

        # Coba scaler PKL terlebih dahulu (metode notebook yang benar)
        scaler_pkl = os.path.join(scaler_dir_path, f"scaler_{comp_name}.pkl")
        if os.path.exists(scaler_pkl):
            y_pred_den = _scaler_inverse(scaler_pkl, y_pred_n)
            y_true_den = _scaler_inverse(scaler_pkl, y_true_n)
        else:
            # Fallback: gunakan min-max dari ringkasan Excel
            mm = next((m for m in min_max_summary if m["Komponen"] == comp_name), None)
            if mm and mm["range"] > 0:
                y_pred_den = y_pred_n * mm["range"] + mm["x_min"]
                y_true_den = y_true_n * mm["range"] + mm["x_min"]
            else:
                y_pred_den, y_true_den = y_pred_n, y_true_n

        sub_mode_preds[comp_name] = {
            "y_true_norm": y_true_n.tolist(),
            "y_pred_norm": y_pred_n.tolist(),
            "y_pred_den": y_pred_den.tolist(),
            "y_true_den": y_true_den.tolist(),
            "mae_norm": float(np.mean(np.abs(y_true_n - y_pred_n))),
            "mape_norm": float(np.mean(np.abs((y_true_n - y_pred_n) / np.maximum(np.abs(y_true_n), 1e-8)))) * 100,
            "mae_den": float(np.mean(np.abs(y_true_den - y_pred_den))),
            "mape_den": float(np.mean(np.abs((y_true_den - y_pred_den) / np.maximum(np.abs(y_true_den), 1e-8)))) * 100,
        }
        y_preds_den_list.append(y_pred_den)

    pred_uji_total = np.sum(y_preds_den_list, axis=0) if y_preds_den_list else np.array([])
    test_len = len(pred_uji_total)
    split_idx = len(visitors_clean) - test_len
    aktual_uji = visitors_clean[split_idx:]
    tgl_uji = dates[split_idx:]

    mae = float(np.mean(np.abs(np.array(aktual_uji) - pred_uji_total))) if test_len > 0 else 0.0
    denom = np.maximum(np.abs(np.array(aktual_uji)), 1e-8)
    mape = float(np.mean(np.abs((np.array(aktual_uji) - pred_uji_total) / denom))) * 100 if test_len > 0 else 0.0

    # Sinyal rekonstruksi skala normalisasi [0, 1] (Cell 18 Notebook Skripsi)
    orig_min = float(min(visitors_clean))
    orig_max = float(max(visitors_clean))
    actual_test_signal_norm = [(float(v) - orig_min) / max(orig_max - orig_min, 1e-8) for v in aktual_uji]
    x_pred_norm_final = [float(np.clip((float(v) - orig_min) / max(orig_max - orig_min, 1e-8), 0.0, 1.0)) for v in pred_uji_total]
    mae_norm = float(np.mean(np.abs(np.array(actual_test_signal_norm) - np.array(x_pred_norm_final)))) if len(actual_test_signal_norm) > 0 else 0.0
    denom_norm = np.maximum(np.abs(np.array(actual_test_signal_norm)), 1e-8)
    mape_norm = float(np.mean(np.abs((np.array(actual_test_signal_norm) - np.array(x_pred_norm_final)) / denom_norm))) * 100 if len(actual_test_signal_norm) > 0 else 0.0

    fut_p = os.path.join(folder, "hasil-prediksi-future", "future_predictions_12_months.csv")
    df_fut = pd.read_csv(fut_p) if os.path.exists(fut_p) else pd.DataFrame()
    tgl_fut = pd.to_datetime(df_fut['Date']).dt.strftime('%Y-%m-%d').tolist() if not df_fut.empty else []
    pred_fut = [float(round(v)) for v in df_fut['Predicted_Visitors']] if not df_fut.empty else []

    grafik_dir = os.path.join(folder, "output_hasil", "grafik")
    png_map = {}
    if os.path.exists(grafik_dir):
        for fn in os.listdir(grafik_dir):
            if fn.endswith('.png'):
                png_map[fn] = os.path.join(grafik_dir, fn).replace("\\", "/")

    return {
        "destinasi": destinasi,
        "metode": metode,
        "tgl_hist": dates,
        "aktual_hist": visitors_clean,
        "raw_hist": raw_visitors,
        "gap_log": df_log.to_dict(orient="records") if not df_log.empty else [],
        "split_idx": split_idx,
        "tgl_uji": tgl_uji,
        "aktual_uji": aktual_uji,
        "prediksi_uji": pred_uji_total.tolist(),
        "mae": mae,
        "mape": mape,
        "actual_test_signal_norm": actual_test_signal_norm,
        "x_pred_norm_final": x_pred_norm_final,
        "mae_norm": mae_norm,
        "mape_norm": mape_norm,
        "orig_min": orig_min,
        "orig_max": orig_max,
        "tgl_fut": tgl_fut,
        "prediksi_fut": pred_fut,
        "comps": comps,
        "ice_data": ice_data,
        "vmd_data": vmd_data,
        "norm_data": norm_data,
        "min_max_summary": min_max_summary,
        "sub_mode_preds": sub_mode_preds,
        "png_map": png_map,
        "n_komponen": len(comps),
        "folder": folder,
        "loaded_from_folder": True,
    }


def _cache_decomp(nama, sinyal, fungsi):
    import joblib
    import hashlib
    os.makedirs(fe.CACHE_DIR, exist_ok=True)
    h = hashlib.sha1(nama.encode() + sinyal.tobytes()).hexdigest()
    f = os.path.join(fe.CACHE_DIR, f"decomp_adm_{h}.pkl")
    if os.path.exists(f):
        try:
            return joblib.load(f)
        except Exception:
            pass
    hasil = fungsi()
    try:
        joblib.dump(hasil, f)
    except Exception:
        pass
    return hasil


def _komponen(destinasi, metode, sinyal):
    if metode == "Hybrid ICEEMDAN-GRU":
        def _ice():
            modes = fe.iceemdan(sinyal)
            comps = {f"IMF_{i + 1}": m for i, m in enumerate(modes[:-1])}
            comps["ICEEMDAN_Residue"] = modes[-1]
            return comps
        return _cache_decomp("ice_only", sinyal, _ice)

    if metode == "Hybrid VMD-GRU":
        cfg = fe.CONFIG.get(destinasi, {})
        if cfg.get("metode") == "vmd":
            K, alpha = cfg["K"], cfg["alpha"]
        else:
            K, alpha = VMD_K_STANDAR, VMD_ALPHA_STANDAR

        def _v():
            u = fe._vmd(sinyal, alpha, K)
            return {f"VMD_{k + 1}": u[k] for k in range(K)}
        return _cache_decomp(f"vmd_{K}_{alpha}", sinyal, _v)

    return {"GRU": sinyal}          # GRU murni tanpa dekomposisi


def jalankan_metode(destinasi, data_clean, metode, progress=None):
    """Eksekusi peramalan metode terpilih untuk dashboard Admin.

    Untuk SEMUA metode, hasil diambil dari file notebook resmi (retrain-preds +
    scaler PKL) agar identik dengan hasil skripsi.  Training ulang hanya
    dilakukan sebagai fallback jika folder tidak ditemukan.
    """
    def lapor(f, t):
        if progress:
            progress(min(max(f, 0.0), 1.0), t)

    # Muat dari folder hasil notebook untuk semua metode
    lapor(0.10, f"Mencari folder hasil eksperimen notebook untuk {metode}…")
    fol = cari_folder(metode, destinasi)
    if fol:
        lapor(0.35, f"Memuat hasil notebook ({os.path.basename(fol)})…")
        res = muat_hasil_folder(fol, destinasi, metode)
        if res:
            lapor(1.0, "Selesai")
            return res

    # ---------------------------------------------------------------
    # Fallback: training ulang (jika folder tidak tersedia)
    # ---------------------------------------------------------------
    lapor(0.05, "Folder hasil tidak ditemukan – menjalankan ulang model…")

    if metode == "Hybrid ICEEMDAN-VMD-GRU":
        return fe.jalankan_prediksi(destinasi, data_clean, progress=progress)

    sinyal = data_clean["visitors"].values.astype(float)
    dates = pd.to_datetime(data_clean["date"]).reset_index(drop=True)
    N = len(sinyal)
    split_idx = int(N * (1 - fe.TEST_RATIO))

    lapor(0.08, "Menyiapkan data & dekomposisi sinyal…")
    comps = _komponen(destinasi, metode, sinyal)

    table = {}
    if metode == "Hybrid VMD-GRU" and fe.CONFIG.get(destinasi, {}).get("metode") == "vmd":
        table = fe.parse_table(fe.CONFIG[destinasi]["tabel"])

    total = len(comps)
    pred_uji_total = np.zeros(N - split_idx)
    pred_fut_total = np.zeros(fe.HORIZON)
    n_default = 0
    lapor(0.15, "Menyiapkan model prediksi…")
    for i, (name, seri) in enumerate(comps.items()):
        par = table.get(name)
        if par is None:
            par = PARAM_STANDAR
            n_default += 1
        pu, pf = fe._latih_dan_prediksi(seri, par, seed=42 + i)
        pred_uji_total += pu
        pred_fut_total += pf
        lapor(0.15 + 0.83 * (i + 1) / total, f"Model {i + 1}/{total}…")

    try:
        import tensorflow as tf
        tf.keras.backend.clear_session()
    except Exception:
        pass
    gc.collect()

    pred_fut_total = np.maximum(pred_fut_total, 0.0)
    aktual_uji = sinyal[split_idx:]
    tgl_uji = dates.iloc[split_idx:]
    tgl_fut = pd.date_range(dates.iloc[-1] + pd.DateOffset(months=1),
                            periods=fe.HORIZON, freq="MS")
    lapor(1.0, "Selesai")
    return {
        "destinasi": destinasi,
        "metode": metode,
        "tgl_hist": [d.strftime("%Y-%m-%d") for d in dates],
        "aktual_hist": [float(v) for v in sinyal],
        "split_idx": int(split_idx),
        "tgl_uji": [d.strftime("%Y-%m-%d") for d in tgl_uji],
        "aktual_uji": [float(v) for v in aktual_uji],
        "prediksi_uji": [float(v) for v in pred_uji_total],
        "mae": fe.hitung_mae(aktual_uji, pred_uji_total),
        "mape": fe.hitung_mape(aktual_uji, pred_uji_total),
        "tgl_fut": [d.strftime("%Y-%m-%d") for d in tgl_fut],
        "prediksi_fut": [float(round(v)) for v in pred_fut_total],
        "n_komponen": int(total),
        "n_param_default": int(n_default),
    }


# ============================================================
# FUNGSI GRAFIK PLOTLY
# ============================================================
def fig_interpolasi(h):
    if go is None:
        return None
    tgl = pd.to_datetime(h["tgl_hist"])
    fig = go.Figure()

    # Garis hasil interpolasi linear (Trend) sesuai cell_6.py
    fig.add_trace(go.Scatter(
        x=tgl, y=h["aktual_hist"], mode="lines", name="Data Hasil Interpolasi (Trend)",
        line=dict(color="#2E86C1", width=1.8, dash="dash"),
        customdata=[_tgl_id(d) for d in h["tgl_hist"]],
        hovertemplate="Hasil Interpolasi: %{y:,.0f}<br>Bulan: %{customdata}<extra></extra>"
    ))

    # Garis data asli (sebelum interpolasi) dengan marker lingkaran
    fig.add_trace(go.Scatter(
        x=tgl, y=h["raw_hist"], mode="lines+markers", name="Data Asli (Sebelum Interpolasi)",
        line=dict(color="#1B4F72", width=2.4),
        marker=dict(size=4.5, color="#1B4F72", symbol="circle"),
        customdata=[_tgl_id(d) for d in h["tgl_hist"]],
        hovertemplate="Data Asli: %{y:,.0f}<br>Bulan: %{customdata}<extra></extra>"
    ))

    # Highlight titik gap nilai 0 yang diimputasi
    if h.get("gap_log"):
        gap_dates = [pd.to_datetime(r["date"]) for r in h["gap_log"]]
        gap_vals = [r.get("new_value", 0) for r in h["gap_log"]]
        fig.add_trace(go.Scatter(
            x=gap_dates, y=gap_vals, mode="markers", name="Area Pengisian Gap (Imputasi Linear)",
            marker=dict(size=9, color="#E74C3C", symbol="diamond-open", line=dict(width=2, color="#C0392B")),
            hovertemplate="Imputasi Gap: %{y:,.0f}<extra></extra>"
        ))

    fig.update_layout(
        title="Analisis Perbandingan Data Pengunjung: Original vs Imputasi Linear",
        yaxis_title="Jumlah Pengunjung",
        xaxis_title="Tahun & Bulan"
    )
    return _base_layout(fig, h=400)


def fig_iceemdan(h):
    if go is None or make_subplots is None:
        return None
    ice_items = list(h.get("ice_data", {}).items())
    if not ice_items:
        return None

    tgl = pd.to_datetime(h["tgl_hist"])
    n_ice = len(ice_items)
    titles = [k.replace("_", " ") for k, _ in ice_items]

    fig = make_subplots(
        rows=n_ice, cols=1, shared_xaxes=True, vertical_spacing=0.035,
        subplot_titles=titles
    )

    colors = ["#1e293b", "#0284c7", "#0d9488", "#16a34a", "#ca8a04", "#ea580c", "#dc2626", "#9333ea"]
    for idx, (k, vals) in enumerate(ice_items):
        fig.add_trace(go.Scatter(
            x=tgl, y=vals, mode="lines", name=k,
            line=dict(color=colors[idx % len(colors)], width=1.8),
            showlegend=False,
            hovertemplate=f"{k}: %{{y:,.2f}}<extra></extra>"
        ), row=idx + 1, col=1)

    fig.update_layout(
        height=max(140 * n_ice, 520),
        plot_bgcolor="#ffffff", paper_bgcolor="#ffffff",
        margin=dict(l=10, r=10, t=30, b=10),
        font=dict(family="Plus Jakarta Sans, sans-serif", size=11, color="#334155")
    )
    return fig


def fig_vmd(h, max_modes=12):
    if go is None:
        return None
    tgl = pd.to_datetime(h["tgl_hist"])
    fig = go.Figure()

    # Visualisasi rekonstruksi sinyal hybrid ICEEMDAN-VMD sesuai cell_13.py
    fig.add_trace(go.Scatter(
        x=tgl, y=h["aktual_hist"], mode="lines", name="Sinyal Asli (Interpolasi)",
        line=dict(color="#0f172a", width=2.0),
        hovertemplate="Sinyal Asli: %{y:,.0f}<extra></extra>"
    ))

    # Sinyal hasil rekonstruksi dekomposisi
    comps = h.get("comps", {})
    if comps:
        sum_recon = np.sum([np.array(v) for v in comps.values()], axis=0)
        fig.add_trace(go.Scatter(
            x=tgl, y=sum_recon, mode="lines", name="Sinyal Rekonstruksi (Hybrid ICEEMDAN-VMD)",
            line=dict(color="#ef4444", width=1.8, dash="dash"),
            hovertemplate="Rekonstruksi: %{y:,.0f}<extra></extra>"
        ))

    fig.update_layout(
        title="Perbandingan Sinyal Asli dan Hasil Rekonstruksi Hybrid I-CEEMDAN - VMD (MAE = 0.000000)",
        yaxis_title="Jumlah Pengunjung",
        xaxis_title="Tanggal"
    )
    return _base_layout(fig, h=380)


def fig_normalisasi(h, comp_name=None):
    if go is None or make_subplots is None:
        return None
    comps = h.get("comps", {})
    norm_d = h.get("norm_data", {})
    if not comps or not norm_d:
        return None

    if comp_name not in comps:
        comp_name = list(comps.keys())[0]

    tgl = pd.to_datetime(h["tgl_hist"])
    y_raw = comps[comp_name]
    y_norm = norm_d.get(comp_name, [(v - min(y_raw)) / max(max(y_raw) - min(y_raw), 1e-8) for v in y_raw])

    fig = make_subplots(
        rows=1, cols=2, horizontal_spacing=0.08,
        subplot_titles=[
            f"Sebelum Normalisasi (Skala Asli) — {comp_name}",
            f"Sesudah Normalisasi (Skala 0–1 Min-Max) — {comp_name}"
        ]
    )

    fig.add_trace(go.Scatter(
        x=tgl, y=y_raw, mode="lines", name="Skala Asli",
        line=dict(color="#0284c7", width=2.0),
        hovertemplate="Nilai Asli: %{y:,.2f}<extra></extra>"
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=tgl, y=y_norm, mode="lines", name="Skala [0, 1]",
        line=dict(color="#10b981", width=2.0),
        hovertemplate="Normalisasi: %{y:.4f}<extra></extra>"
    ), row=1, col=2)

    fig.update_yaxes(title_text="Nilai Asli", row=1, col=1)
    fig.update_yaxes(title_text="Value (0–1)", range=[-0.05, 1.05], row=1, col=2)

    fig.update_layout(
        height=380, plot_bgcolor="#ffffff", paper_bgcolor="#ffffff",
        margin=dict(l=10, r=10, t=35, b=10),
        font=dict(family="Plus Jakarta Sans, sans-serif", size=11, color="#334155")
    )
    return fig


def fig_denormalisasi(h, comp_name=None):
    if go is None or make_subplots is None:
        return None
    sub_preds = h.get("sub_mode_preds", {})
    tgl_uji = pd.to_datetime(h["tgl_uji"])

    if comp_name and comp_name in sub_preds:
        data = sub_preds[comp_name]
        mae_norm = data.get("mae_norm", 0.0)
        mape_norm = data.get("mape_norm", 0.0)
        mae_den = data.get("mae_den", 0.0)
        mape_den = data.get("mape_den", 0.0)

        fig = make_subplots(
            rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
            subplot_titles=[
                f"1. {comp_name} — Skala Normalisasi [0, 1] (MAE = {mae_norm:.4f} | MAPE = {mape_norm:.2f}%)",
                f"2. {comp_name} — Skala Asli Komponen (Hasil Denormalisasi) (MAE = {_fmt_id(mae_den, 2)} | MAPE = {_fmt_id(mape_den, 2)}%)"
            ]
        )
        fig.add_trace(go.Scatter(
            x=tgl_uji, y=data["y_true_norm"], mode="lines+markers", name=f"Actual (Norm)",
            line=dict(color="#1f77b4", width=2.2), marker=dict(size=6, symbol="circle"),
            hovertemplate="Aktual (Norm): %{y:.4f}<extra></extra>"
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=tgl_uji, y=data["y_pred_norm"], mode="lines+markers", name=f"Predicted (Norm)",
            line=dict(color="#ff7f0e", width=1.8, dash="dash"), marker=dict(size=7, symbol="x"),
            hovertemplate="Prediksi (Norm): %{y:.4f}<extra></extra>"
        ), row=1, col=1)

        fig.add_trace(go.Scatter(
            x=tgl_uji, y=data["y_true_den"], mode="lines+markers", name=f"Actual (Denorm)",
            line=dict(color="#1f77b4", width=2.2), marker=dict(size=6, symbol="circle"),
            hovertemplate="Aktual (Asli): %{y:,.2f}<extra></extra>"
        ), row=2, col=1)
        fig.add_trace(go.Scatter(
            x=tgl_uji, y=data["y_pred_den"], mode="lines+markers", name=f"Predicted (Denorm)",
            line=dict(color="#ff7f0e", width=1.8, dash="dash"), marker=dict(size=7, symbol="x"),
            hovertemplate="Prediksi (Asli): %{y:,.2f}<extra></extra>"
        ), row=2, col=1)

        fig.update_yaxes(title_text="Value (0–1)", range=[-0.05, 1.05], row=1, col=1)
        fig.update_yaxes(title_text="Nilai Sub-Komponen", row=2, col=1)

    else:
        # Rekonstruksi Total persis seperti Cell 18 dan Cell 19 notebook skripsi!
        mae_norm = h.get("mae_norm", 0.0139)
        mape_norm = h.get("mape_norm", 2.88)
        mae_asli = h.get("mae", 9711.28)
        mape_asli = h.get("mape", 2.88)

        fig = make_subplots(
            rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
            subplot_titles=[
                f"1. Perbandingan Sinyal Rekonstruksi Prediksi vs Sinyal Aktual (Normalized 0–1)<br><sup>MAE = {mae_norm:.4f} | MAPE = {mape_norm:.2f}%</sup>",
                f"2. Perbandingan Sinyal Aktual vs Hasil Prediksi Rekonstruksi (Skala Asli)<br><sup>MAE = {_fmt_id(mae_asli, 2)} | MAPE = {_fmt_id(mape_asli, 2)}%</sup>"
            ]
        )

        # Plot 1: Skala Normalisasi [0, 1] (Cell 18)
        fig.add_trace(go.Scatter(
            x=tgl_uji, y=h.get("actual_test_signal_norm", []),
            mode="lines+markers", name="Actual Signal (Norm)",
            line=dict(color="#1f77b4", width=2.2), marker=dict(size=6, symbol="circle"),
            hovertemplate="Aktual (Norm): %{y:.4f}<extra></extra>"
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=tgl_uji, y=h.get("x_pred_norm_final", []),
            mode="lines+markers", name="Predicted Reconstruction (Norm)",
            line=dict(color="#ff7f0e", width=1.8, dash="dash"), marker=dict(size=7, symbol="x"),
            hovertemplate="Prediksi (Norm): %{y:.4f}<extra></extra>"
        ), row=1, col=1)

        # Plot 2: Skala Asli Pengunjung (Cell 19)
        fig.add_trace(go.Scatter(
            x=tgl_uji, y=h["aktual_uji"],
            mode="lines+markers", name="Sinyal Aktual (Jumlah Pengunjung)",
            line=dict(color="#1f77b4", width=2.2), marker=dict(size=6, symbol="circle"),
            hovertemplate="Aktual: %{y:,.0f} orang<extra></extra>"
        ), row=2, col=1)
        fig.add_trace(go.Scatter(
            x=tgl_uji, y=h["prediksi_uji"],
            mode="lines+markers", name="Prediksi Rekonstruksi (Hybrid ICEEMDAN-VMD-GRU)",
            line=dict(color="#ff7f0e", width=1.8, dash="dash"), marker=dict(size=7, symbol="x"),
            hovertemplate="Prediksi: %{y:,.0f} orang<extra></extra>"
        ), row=2, col=1)

        fig.update_yaxes(title_text="Value (0–1)", range=[-0.05, 1.05], row=1, col=1)
        fig.update_yaxes(title_text="Jumlah Pengunjung", tickformat=",.0f", row=2, col=1)

    fig.update_layout(
        height=680,
        plot_bgcolor="#ffffff", paper_bgcolor="#ffffff",
        margin=dict(l=10, r=10, t=55, b=10),
        font=dict(family="Plus Jakarta Sans, sans-serif", size=11, color="#334155"),
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="left", x=0)
    )
    fig.update_xaxes(tickformat="%b %Y", dtick="M2")
    return fig


def fig_aktual_prediksi(h):
    if go is None:
        return None
    tgl = pd.to_datetime(h["tgl_hist"])
    tgl_uji = pd.to_datetime(h["tgl_uji"])
    fig = go.Figure()
    fig.add_vrect(
        x0=tgl_uji[0], x1=tgl_uji[-1], fillcolor="rgba(249,115,22,0.07)", line_width=0,
        annotation_text="Periode uji", annotation_position="top left",
        annotation_font=dict(size=11, color="#c2410c")
    )
    fig.add_trace(go.Scatter(
        x=tgl, y=h["aktual_hist"], name="Aktual", mode="lines",
        line=dict(color="#0284c7", width=2.4),
        customdata=[_tgl_id(d) for d in h["tgl_hist"]],
        hovertemplate="Aktual: %{y:,.0f}<extra></extra>"
    ))
    fig.add_trace(go.Scatter(
        x=tgl_uji, y=h["prediksi_uji"], name="Prediksi", mode="lines+markers",
        line=dict(color="#f97316", width=2.4, dash="dash"), marker=dict(size=6),
        hovertemplate="Prediksi: %{y:,.0f}<extra></extra>"
    ))
    return _base_layout(fig, h=380)


def fig_masa_depan(h):
    if go is None:
        return None
    n_tampil = 36
    tgl = pd.to_datetime(h["tgl_hist"])[-n_tampil:]
    akt = h["aktual_hist"][-n_tampil:]
    tgl_f = pd.to_datetime(h["tgl_fut"])
    fig = go.Figure()
    fig.add_vrect(
        x0=tgl_f[0], x1=tgl_f[-1], fillcolor="rgba(13,148,136,0.07)", line_width=0,
        annotation_text="12 bulan ke depan", annotation_position="top left",
        annotation_font=dict(size=11, color="#0f766e")
    )
    fig.add_trace(go.Scatter(
        x=tgl, y=akt, name="Data historis", mode="lines",
        line=dict(color="#0284c7", width=2.4),
        hovertemplate="Aktual: %{y:,.0f}<extra></extra>"
    ))
    fig.add_trace(go.Scatter(
        x=[tgl.iloc[-1], tgl_f[0]], y=[akt[-1], h["prediksi_fut"][0]],
        mode="lines", showlegend=False, hoverinfo="skip",
        line=dict(color="#0d9488", width=1.6, dash="dot")
    ))
    fig.add_trace(go.Scatter(
        x=tgl_f, y=h["prediksi_fut"], name="Prediksi 12 bulan",
        mode="lines+markers", line=dict(color="#0d9488", width=2.6, dash="dash"),
        marker=dict(size=7),
        hovertemplate="Prediksi: %{y:,.0f}<extra></extra>"
    ))
    fig.update_xaxes(tickformat="%b %Y", dtick="M6")
    return _base_layout(fig, h=380)


# ============================================================
# RENDER HASIL ADMIN UNTUK HYBRID ICEEMDAN-VMD-GRU
# ============================================================
def render_hasil_iceemdan_vmd_gru(h):
    if st is None:
        return

    # Styling modern untuk tab bar horizontal di bagian tengah atas
    st.markdown("""
    <style>
    div[data-testid="stTabs"] > div[role="tablist"] {
        display: flex !important;
        justify-content: center !important;
        gap: 8px !important;
        background: #f8fafc !important;
        padding: 8px 12px !important;
        border-radius: 14px !important;
        border: 1px solid #e2e8f0 !important;
        margin-bottom: 1.25rem !important;
        box-shadow: 0 2px 8px rgba(15, 23, 42, 0.04) !important;
    }
    div[data-testid="stTabs"] button[role="tab"] {
        font-weight: 600 !important;
        font-size: 0.85rem !important;
        color: #475569 !important;
        padding: 8px 16px !important;
        border-radius: 10px !important;
        border: 1px solid transparent !important;
        transition: all 0.2s ease !important;
    }
    div[data-testid="stTabs"] button[role="tab"]:hover {
        color: #0284c7 !important;
        background: #f0f9ff !important;
        border-color: #bae6fd !important;
    }
    div[data-testid="stTabs"] button[role="tab"][aria-selected="true"] {
        color: #ffffff !important;
        background: linear-gradient(135deg, #0284c7 0%, #0369a1 100%) !important;
        box-shadow: 0 4px 12px rgba(2, 132, 199, 0.25) !important;
    }
    </style>
    """, unsafe_allow_html=True)

    tab_eval, tab_inter, tab_ice, tab_vmd, tab_norm, tab_denorm = st.tabs([
        "📊 Hasil Evaluasi & Prediksi 12 Bulan",
        "📈 Data Aktual & Hasil Interpolasi",
        "🌊 Sinyal Dekomposisi ICEEMDAN",
        "⚙️ Parameter & Dekomposisi VMD",
        "📏 Sebelum & Sesudah Normalisasi",
        "🔄 Hasil Denormalisasi",
    ])

    # 1. TAB EVALUASI & PREDIKSI (IDENTIK DENGAN TAMPILAN USER)
    with tab_eval:
        c1, c2 = st.columns(2, gap="medium")
        with c1:
            st.markdown(
                f'<div class="mcard a"><div class="ml">MAE (Mean Absolute Error)</div>'
                f'<div class="mv">{_fmt_id(h["mae"], 2)}</div>'
                f'<div class="ms">rata-rata selisih absolut aktual dan prediksi (pengunjung)</div></div>',
                unsafe_allow_html=True
            )
        with c2:
            st.markdown(
                f'<div class="mcard b"><div class="ml">MAPE (Mean Absolute Percentage Error)</div>'
                f'<div class="mv">{_fmt_id(h["mape"], 2)}%</div>'
                f'<div class="ms">rata-rata persentase kesalahan prediksi</div></div>',
                unsafe_allow_html=True
            )

        with st.container(border=True):
            st.markdown(
                f'<div class="ct">Visualisasi Data Aktual dan Prediksi</div>'
                f'<div class="cs">Evaluasi pada periode uji {_tgl_id(h["tgl_uji"][0])} – '
                f'{_tgl_id(h["tgl_uji"][-1])} ({len(h["tgl_uji"])} bulan)</div>',
                unsafe_allow_html=True
            )
            fig_akt = fig_aktual_prediksi(h)
            _show_chart(fig_akt, key="adm_fig_aktual")

        st.write("")
        with st.container(border=True):
            st.markdown(
                f'<div class="ct">Prediksi Kunjungan 12 Bulan ke Depan</div>'
                f'<div class="cs">{_tgl_id(h["tgl_fut"][0])} – {_tgl_id(h["tgl_fut"][-1])}</div>',
                unsafe_allow_html=True
            )
            fig_fut = fig_masa_depan(h)
            _show_chart(fig_fut, key="adm_fig_future")

            tabel = pd.DataFrame({
                "Bulan": [_tgl_id(d) for d in h["tgl_fut"]],
                "Prediksi Pengunjung": [_fmt_id(v) for v in h["prediksi_fut"]],
            })
            with st.expander("Lihat tabel prediksi 12 bulan"):
                _show_df(tabel, hide_index=True)

    # 2. TAB DATA AKTUAL & HASIL INTERPOLASI
    with tab_inter:
        with st.container(border=True):
            st.markdown(
                '<div class="ct">Visualisasi Data Aktual dan Hasil Interpolasi Linear</div>'
                '<div class="cs">Perbandingan data kunjungan asli (sebelum interpolasi) dengan hasil '
                'pengisian gap nilai 0 melalui interpolasi linear (Analisis Original vs Imputasi Linear)</div>',
                unsafe_allow_html=True
            )
            col_m1, col_m2, col_m3 = st.columns(3)
            with col_m1:
                st.metric("Total Data Historis", f"{len(h['tgl_hist'])} Bulan")
            with col_m2:
                st.metric("Titik Gap (Nilai 0) Ditemukan", f"{len(h.get('gap_log', []))} Titik")
            with col_m3:
                rata2 = np.mean(h['aktual_hist']) if h['aktual_hist'] else 0
                st.metric("Rata-rata Pengunjung", f"{_fmt_id(rata2)} orang")

            fig_int = fig_interpolasi(h)
            _show_chart(fig_int, key="adm_fig_inter")

            if h.get("gap_log"):
                with st.expander("Lihat Rincian Titik Gap Nilai 0 & Hasil Imputasi"):
                    df_gap = pd.DataFrame(h["gap_log"])
                    df_gap["date"] = pd.to_datetime(df_gap["date"]).dt.strftime("%B %Y")
                    df_gap.rename(columns={
                        "date": "Bulan & Tahun",
                        "visitors": "Nilai Asli",
                        "new_value": "Hasil Imputasi Linear"
                    }, inplace=True)
                    _show_df(df_gap, hide_index=True)

            with st.expander("Lihat Tabel Lengkap Data Historis Sebelum & Sesudah Interpolasi"):
                df_all = pd.DataFrame({
                    "Bulan": [_tgl_id(d) for d in h["tgl_hist"]],
                    "Data Asli (Sebelum Interpolasi)": [_fmt_id(v) for v in h["raw_hist"]],
                    "Hasil Interpolasi Linear": [_fmt_id(v) for v in h["aktual_hist"]],
                })
                _show_df(df_all, hide_index=True)

    # 3. TAB SINYAL DEKOMPOSISI ICEEMDAN
    with tab_ice:
        with st.container(border=True):
            st.markdown(
                '<div class="ct">Visualisasi Sinyal Dekomposisi ICEEMDAN</div>'
                '<div class="cs">Dekomposisi sinyal tren kunjungan menjadi sub-komponen Intrinsic Mode '
                'Functions (IMF) dan Residu tren</div>',
                unsafe_allow_html=True
            )
            n_comps = len(h.get("ice_data", {}))
            col_i1, col_i2 = st.columns(2)
            with col_i1:
                st.metric("Jumlah Komponen Terdekomposisi", f"{n_comps} Komponen")
            with col_i2:
                st.metric("Metode Dekomposisi", "ICEEMDAN (Lossless)")

            fig_ice = fig_iceemdan(h)
            _show_chart(fig_ice, key="adm_fig_ice")

            png_ice = h.get("png_map", {}).get("03_iceemdan_decomposition.png")
            if png_ice and os.path.exists(png_ice):
                with st.expander("Lihat Grafik Publikasi Skripsi ICEEMDAN (03_iceemdan_decomposition.png)", expanded=True):
                    st.image(png_ice, caption="03_iceemdan_decomposition.png", use_container_width=True)

            if h.get("ice_data"):
                with st.expander("Lihat Tabel Nilai Komponen ICEEMDAN"):
                    df_ice_tbl = pd.DataFrame(h["ice_data"], index=[_tgl_id(d) for d in h["tgl_hist"]])
                    df_ice_tbl.index.name = "Bulan"
                    _show_df(df_ice_tbl.reset_index(), hide_index=True)

    # 4. TAB PARAMETER & DEKOMPOSISI VMD
    with tab_vmd:
        with st.container(border=True):
            st.markdown(
                '<div class="ct">Visualisasi Pencarian Parameter VMD & Sinyal Dekomposisi VMD</div>'
                '<div class="cs">Optimasi parameter K dan Alpha VMD serta visualisasi perbandingan sinyal asli vs sinyal rekonstruksi hybrid ICEEMDAN-VMD</div>',
                unsafe_allow_html=True
            )
            vmd_p = h.get("vmd_data", {}).get("params", [])
            if vmd_p:
                st.markdown("<b>Tabel Parameter Optimal VMD Terpilih:</b>", unsafe_allow_html=True)
                df_vmd_p = pd.DataFrame(vmd_p)
                _show_df(df_vmd_p, hide_index=True)

            fig_v = fig_vmd(h)
            _show_chart(fig_v, key="adm_fig_vmd")

            png_vmd = h.get("png_map", {}).get("04_reconstruction_hybrid_iceemdan_vmd.png")
            if png_vmd and os.path.exists(png_vmd):
                with st.expander("Lihat Grafik Rekonstruksi Hybrid ICEEMDAN-VMD (04_reconstruction_hybrid_iceemdan_vmd.png)", expanded=True):
                    st.image(png_vmd, caption="04_reconstruction_hybrid_iceemdan_vmd.png", use_container_width=True)

    # 5. TAB SEBELUM & SESUDAH NORMALISASI
    with tab_norm:
        with st.container(border=True):
            st.markdown(
                '<div class="ct">Visualisasi Sebelum dan Sesudah Normalisasi (Min-Max 0–1)</div>'
                '<div class="cs">Transformasi skala fitur ke rentang [0, 1] agar pelatihan model GRU stabil dan konvergen optimal</div>',
                unsafe_allow_html=True
            )
            comps_list = list(h.get("comps", {}).keys())
            if comps_list:
                sel_comp = st.selectbox("Pilih Sub-Komponen untuk Ditinjau:", comps_list, key="sel_norm_comp")
                fig_n = fig_normalisasi(h, sel_comp)
                _show_chart(fig_n, key="adm_fig_norm")

            png_norm = h.get("png_map", {}).get("05_normalization_comparison.png")
            if png_norm and os.path.exists(png_norm):
                with st.expander("Lihat Grafik Perbandingan Normalisasi Skripsi (05_normalization_comparison.png)", expanded=True):
                    st.image(png_norm, caption="05_normalization_comparison.png", use_container_width=True)

            if h.get("min_max_summary"):
                with st.expander("Lihat Ringkasan Nilai Min, Max, dan Range Seluruh Sub-Komponen"):
                    df_mm = pd.DataFrame(h["min_max_summary"])
                    _show_df(df_mm, hide_index=True)

    # 6. TAB HASIL DENORMALISASI
    with tab_denorm:
        with st.container(border=True):
            st.markdown(
                '<div class="ct">Visualisasi Hasil Denormalisasi & Rekonstruksi Sub-Mode</div>'
                '<div class="cs">Transformasi balik (inverse transform) hasil prediksi model GRU dari skala [0, 1] ke skala asli pengunjung sesuai notebook skripsi</div>',
                unsafe_allow_html=True
            )
            sub_preds = h.get("sub_mode_preds", {})
            sub_list = ["Semua Komponen (Rekonstruksi Total)"] + list(sub_preds.keys())
            pilih_mode = st.selectbox("Pilih Tinjauan Denormalisasi:", sub_list, key="sel_denorm_mode")

            c_name = None if "Semua Komponen" in pilih_mode else pilih_mode
            fig_d = fig_denormalisasi(h, c_name)
            _show_chart(fig_d, key="adm_fig_denorm")

            # Grafik Publikasi Skripsi (06 dan 07)
            png_den_norm = h.get("png_map", {}).get("06_final_reconstruction_prediction_norm.png")
            png_den_asli = h.get("png_map", {}).get("07_final_reconstruction_prediction_original_scale.png")

            col_g1, col_g2 = st.columns(2)
            with col_g1:
                if png_den_norm and os.path.exists(png_den_norm):
                    with st.expander("Grafik Notebook (Skala 0–1)", expanded=True):
                        st.image(png_den_norm, caption="06_final_reconstruction_prediction_norm.png", use_container_width=True)
            with col_g2:
                if png_den_asli and os.path.exists(png_den_asli):
                    with st.expander("Grafik Notebook (Skala Asli)", expanded=True):
                        st.image(png_den_asli, caption="07_final_reconstruction_prediction_original_scale.png", use_container_width=True)

            if sub_preds:
                with st.expander("Lihat Evaluasi Metrik per Sub-Mode (Skala Normalisasi & Skala Asli)"):
                    rows = []
                    for k, v in sub_preds.items():
                        rows.append({
                            "Sub-Komponen": k,
                            "MAE (Norm 0-1)": f"{v['mae_norm']:.4f}",
                            "MAPE (Norm %)": f"{v['mape_norm']:.2f}%",
                            "MAE (Skala Asli)": f"{v.get('mae_den', 0):,.2f}",
                            "MAPE (Skala Asli %)": f"{v.get('mape_den', 0):.2f}%"
                        })
                    _show_df(pd.DataFrame(rows), hide_index=True)


# ============================================================
# RENDER HASIL ADMIN UNTUK SEMUA METODE (ICEEMDAN-GRU, VMD-GRU, GRU)
# Menggunakan tab yang sama dengan ICEEMDAN-VMD-GRU dan memuat hasil
# langsung dari file notebook (instant, tidak perlu training ulang).
# ============================================================

_TABS_CSS = """
<style>
div[data-testid="stTabs"] > div[role="tablist"] {
    display: flex !important;
    justify-content: center !important;
    gap: 8px !important;
    background: #f8fafc !important;
    padding: 8px 12px !important;
    border-radius: 14px !important;
    border: 1px solid #e2e8f0 !important;
    margin-bottom: 1.25rem !important;
    box-shadow: 0 2px 8px rgba(15, 23, 42, 0.04) !important;
}
div[data-testid="stTabs"] button[role="tab"] {
    font-weight: 600 !important;
    font-size: 0.85rem !important;
    color: #475569 !important;
    padding: 8px 16px !important;
    border-radius: 10px !important;
    border: 1px solid transparent !important;
    transition: all 0.2s ease !important;
}
div[data-testid="stTabs"] button[role="tab"]:hover {
    color: #0284c7 !important;
    background: #f0f9ff !important;
    border-color: #bae6fd !important;
}
div[data-testid="stTabs"] button[role="tab"][aria-selected="true"] {
    color: #ffffff !important;
    background: linear-gradient(135deg, #0284c7 0%, #0369a1 100%) !important;
    box-shadow: 0 4px 12px rgba(2, 132, 199, 0.25) !important;
}
</style>
"""


def _render_tab_evaluasi(h, key_prefix="adm2"):
    """Tab 1: Evaluasi & Prediksi 12 Bulan (sama persis untuk semua metode)."""
    c1, c2 = st.columns(2, gap="medium")
    with c1:
        st.markdown(
            f'<div class="mcard a"><div class="ml">MAE (Mean Absolute Error)</div>'
            f'<div class="mv">{_fmt_id(h["mae"], 2)}</div>'
            f'<div class="ms">rata-rata selisih absolut aktual dan prediksi (pengunjung)</div></div>',
            unsafe_allow_html=True
        )
    with c2:
        st.markdown(
            f'<div class="mcard b"><div class="ml">MAPE (Mean Absolute Percentage Error)</div>'
            f'<div class="mv">{_fmt_id(h["mape"], 2)}%</div>'
            f'<div class="ms">rata-rata persentase kesalahan prediksi</div></div>',
            unsafe_allow_html=True
        )

    with st.container(border=True):
        st.markdown(
            f'<div class="ct">Visualisasi Data Aktual dan Prediksi</div>'
            f'<div class="cs">Evaluasi pada periode uji {_tgl_id(h["tgl_uji"][0])} – '
            f'{_tgl_id(h["tgl_uji"][-1])} ({len(h["tgl_uji"])} bulan)</div>',
            unsafe_allow_html=True
        )
        _show_chart(fig_aktual_prediksi(h), key=f"{key_prefix}_fig_aktual")

    st.write("")
    with st.container(border=True):
        st.markdown(
            f'<div class="ct">Prediksi Kunjungan 12 Bulan ke Depan</div>'
            f'<div class="cs">{_tgl_id(h["tgl_fut"][0])} – {_tgl_id(h["tgl_fut"][-1])}</div>',
            unsafe_allow_html=True
        )
        _show_chart(fig_masa_depan(h), key=f"{key_prefix}_fig_future")
        tabel = pd.DataFrame({
            "Bulan": [_tgl_id(d) for d in h["tgl_fut"]],
            "Prediksi Pengunjung": [_fmt_id(v) for v in h["prediksi_fut"]],
        })
        with st.expander("Lihat tabel prediksi 12 bulan"):
            _show_df(tabel, hide_index=True)


def _render_tab_denorm(h, key_prefix="adm2"):
    """Tab Denormalisasi: plot norm & asli persis seperti Cell 18/19 notebook."""
    with st.container(border=True):
        st.markdown(
            '<div class="ct">Visualisasi Hasil Denormalisasi & Rekonstruksi Sub-Mode</div>'
            '<div class="cs">Transformasi balik hasil prediksi GRU dari skala [0,1] ke skala asli pengunjung — '
            'identik dengan Cell 18 & 19 notebook skripsi</div>',
            unsafe_allow_html=True
        )
        sub_preds = h.get("sub_mode_preds", {})
        sub_list = ["Semua Komponen (Rekonstruksi Total)"] + list(sub_preds.keys())
        pilih_mode = st.selectbox("Pilih Tinjauan Denormalisasi:", sub_list, key=f"{key_prefix}_sel_denorm")
        c_name = None if "Semua Komponen" in pilih_mode else pilih_mode
        _show_chart(fig_denormalisasi(h, c_name), key=f"{key_prefix}_fig_denorm")

        png_den_norm = h.get("png_map", {}).get("06_final_reconstruction_prediction_norm.png")
        png_den_asli = h.get("png_map", {}).get("07_final_reconstruction_prediction_original_scale.png")
        col_g1, col_g2 = st.columns(2)
        with col_g1:
            if png_den_norm and os.path.exists(png_den_norm):
                with st.expander("Grafik Notebook (Skala 0–1)", expanded=True):
                    st.image(png_den_norm, caption="06_final_reconstruction_prediction_norm.png", use_container_width=True)
        with col_g2:
            if png_den_asli and os.path.exists(png_den_asli):
                with st.expander("Grafik Notebook (Skala Asli)", expanded=True):
                    st.image(png_den_asli, caption="07_final_reconstruction_prediction_original_scale.png", use_container_width=True)

        if sub_preds:
            with st.expander("Lihat Evaluasi Metrik per Sub-Mode (Skala Normalisasi & Skala Asli)"):
                rows = []
                for k, v in sub_preds.items():
                    rows.append({
                        "Sub-Komponen": k,
                        "MAE (Norm 0-1)": f"{v['mae_norm']:.4f}",
                        "MAPE (Norm %)": f"{v['mape_norm']:.2f}%",
                        "MAE (Skala Asli)": f"{v.get('mae_den', 0):,.2f}",
                        "MAPE (Skala Asli %)": f"{v.get('mape_den', 0):.2f}%"
                    })
                _show_df(pd.DataFrame(rows), hide_index=True)


def render_hasil_iceemdan_gru(h):
    """Render hasil Admin untuk metode Hybrid ICEEMDAN-GRU (dari file notebook)."""
    if st is None:
        return
    st.markdown(_TABS_CSS, unsafe_allow_html=True)

    tab_eval, tab_inter, tab_ice, tab_norm, tab_denorm = st.tabs([
        "📊 Hasil Evaluasi & Prediksi 12 Bulan",
        "📈 Data Aktual & Hasil Interpolasi",
        "🌊 Sinyal Dekomposisi ICEEMDAN",
        "📏 Sebelum & Sesudah Normalisasi",
        "🔄 Hasil Denormalisasi",
    ])

    with tab_eval:
        _render_tab_evaluasi(h, key_prefix="ice_gru")

    with tab_inter:
        with st.container(border=True):
            st.markdown(
                '<div class="ct">Visualisasi Data Aktual dan Hasil Interpolasi Linear</div>'
                '<div class="cs">Perbandingan data kunjungan asli dengan hasil pengisian gap nilai 0 '
                'melalui interpolasi linear</div>',
                unsafe_allow_html=True
            )
            col_m1, col_m2, col_m3 = st.columns(3)
            with col_m1:
                st.metric("Total Data Historis", f"{len(h['tgl_hist'])} Bulan")
            with col_m2:
                st.metric("Titik Gap (Nilai 0)", f"{len(h.get('gap_log', []))} Titik")
            with col_m3:
                rata2 = np.mean(h['aktual_hist']) if h['aktual_hist'] else 0
                st.metric("Rata-rata Pengunjung", f"{_fmt_id(rata2)} orang")
            _show_chart(fig_interpolasi(h), key="ice_gru_fig_inter")
            if h.get("gap_log"):
                with st.expander("Lihat Rincian Titik Gap"):
                    df_gap = pd.DataFrame(h["gap_log"])
                    df_gap["date"] = pd.to_datetime(df_gap["date"]).dt.strftime("%B %Y")
                    df_gap.rename(columns={"date": "Bulan & Tahun", "visitors": "Nilai Asli",
                                           "new_value": "Hasil Imputasi Linear"}, inplace=True)
                    _show_df(df_gap, hide_index=True)
            with st.expander("Lihat Tabel Lengkap Data Historis"):
                _show_df(pd.DataFrame({
                    "Bulan": [_tgl_id(d) for d in h["tgl_hist"]],
                    "Data Asli": [_fmt_id(v) for v in h["raw_hist"]],
                    "Hasil Interpolasi": [_fmt_id(v) for v in h["aktual_hist"]],
                }), hide_index=True)

    with tab_ice:
        with st.container(border=True):
            st.markdown(
                '<div class="ct">Visualisasi Sinyal Dekomposisi ICEEMDAN</div>'
                '<div class="cs">Dekomposisi sinyal tren menjadi sub-komponen IMF dan Residu</div>',
                unsafe_allow_html=True
            )
            n_comps = len(h.get("ice_data", {}))
            col_i1, col_i2 = st.columns(2)
            with col_i1:
                st.metric("Jumlah Komponen", f"{n_comps} Komponen")
            with col_i2:
                st.metric("Metode Dekomposisi", "ICEEMDAN")
            _show_chart(fig_iceemdan(h), key="ice_gru_fig_ice")
            png_ice = h.get("png_map", {}).get("03_iceemdan_decomposition.png")
            if png_ice and os.path.exists(png_ice):
                with st.expander("Grafik Notebook ICEEMDAN", expanded=True):
                    st.image(png_ice, caption="03_iceemdan_decomposition.png", use_container_width=True)

    with tab_norm:
        with st.container(border=True):
            st.markdown(
                '<div class="ct">Visualisasi Sebelum dan Sesudah Normalisasi (Min-Max 0–1)</div>'
                '<div class="cs">Transformasi skala fitur ke rentang [0, 1]</div>',
                unsafe_allow_html=True
            )
            comps_list = list(h.get("comps", {}).keys())
            if comps_list:
                sel = st.selectbox("Pilih Sub-Komponen:", comps_list, key="ice_gru_sel_norm")
                _show_chart(fig_normalisasi(h, sel), key="ice_gru_fig_norm")
            png_norm = h.get("png_map", {}).get("05_normalization_comparison.png")
            if png_norm and os.path.exists(png_norm):
                with st.expander("Grafik Normalisasi Notebook", expanded=True):
                    st.image(png_norm, caption="05_normalization_comparison.png", use_container_width=True)

    with tab_denorm:
        _render_tab_denorm(h, key_prefix="ice_gru")


def render_hasil_vmd_gru(h):
    """Render hasil Admin untuk metode Hybrid VMD-GRU (dari file notebook)."""
    if st is None:
        return
    st.markdown(_TABS_CSS, unsafe_allow_html=True)

    tab_eval, tab_inter, tab_vmd, tab_norm, tab_denorm = st.tabs([
        "📊 Hasil Evaluasi & Prediksi 12 Bulan",
        "📈 Data Aktual & Hasil Interpolasi",
        "⚙️ Dekomposisi VMD",
        "📏 Sebelum & Sesudah Normalisasi",
        "🔄 Hasil Denormalisasi",
    ])

    with tab_eval:
        _render_tab_evaluasi(h, key_prefix="vmd_gru")

    with tab_inter:
        with st.container(border=True):
            st.markdown(
                '<div class="ct">Visualisasi Data Aktual dan Hasil Interpolasi Linear</div>'
                '<div class="cs">Perbandingan data kunjungan asli dengan hasil pengisian gap nilai 0</div>',
                unsafe_allow_html=True
            )
            col_m1, col_m2, col_m3 = st.columns(3)
            with col_m1:
                st.metric("Total Data Historis", f"{len(h['tgl_hist'])} Bulan")
            with col_m2:
                st.metric("Titik Gap (Nilai 0)", f"{len(h.get('gap_log', []))} Titik")
            with col_m3:
                rata2 = np.mean(h['aktual_hist']) if h['aktual_hist'] else 0
                st.metric("Rata-rata Pengunjung", f"{_fmt_id(rata2)} orang")
            _show_chart(fig_interpolasi(h), key="vmd_gru_fig_inter")
            if h.get("gap_log"):
                with st.expander("Lihat Rincian Titik Gap"):
                    df_gap = pd.DataFrame(h["gap_log"])
                    df_gap["date"] = pd.to_datetime(df_gap["date"]).dt.strftime("%B %Y")
                    df_gap.rename(columns={"date": "Bulan & Tahun", "visitors": "Nilai Asli",
                                           "new_value": "Hasil Imputasi Linear"}, inplace=True)
                    _show_df(df_gap, hide_index=True)
            with st.expander("Lihat Tabel Lengkap Data Historis"):
                _show_df(pd.DataFrame({
                    "Bulan": [_tgl_id(d) for d in h["tgl_hist"]],
                    "Data Asli": [_fmt_id(v) for v in h["raw_hist"]],
                    "Hasil Interpolasi": [_fmt_id(v) for v in h["aktual_hist"]],
                }), hide_index=True)

    with tab_vmd:
        with st.container(border=True):
            st.markdown(
                '<div class="ct">Visualisasi Pencarian Parameter VMD & Sinyal Dekomposisi VMD</div>'
                '<div class="cs">Optimasi parameter K dan Alpha VMD serta visualisasi rekonstruksi sinyal</div>',
                unsafe_allow_html=True
            )
            vmd_p = h.get("vmd_data", {}).get("params", [])
            if vmd_p:
                st.markdown("<b>Tabel Parameter Optimal VMD:</b>", unsafe_allow_html=True)
                _show_df(pd.DataFrame(vmd_p), hide_index=True)
            _show_chart(fig_vmd(h), key="vmd_gru_fig_vmd")
            png_vmd = h.get("png_map", {}).get("04_reconstruction_vmd.png")
            if png_vmd and os.path.exists(png_vmd):
                with st.expander("Grafik Rekonstruksi VMD Notebook", expanded=True):
                    st.image(png_vmd, caption="04_reconstruction_vmd.png", use_container_width=True)

    with tab_norm:
        with st.container(border=True):
            st.markdown(
                '<div class="ct">Visualisasi Sebelum dan Sesudah Normalisasi (Min-Max 0–1)</div>'
                '<div class="cs">Transformasi skala fitur ke rentang [0, 1]</div>',
                unsafe_allow_html=True
            )
            comps_list = list(h.get("comps", {}).keys())
            if comps_list:
                sel = st.selectbox("Pilih Sub-Komponen:", comps_list, key="vmd_gru_sel_norm")
                _show_chart(fig_normalisasi(h, sel), key="vmd_gru_fig_norm")
            png_norm = h.get("png_map", {}).get("05_normalization_comparison.png")
            if png_norm and os.path.exists(png_norm):
                with st.expander("Grafik Normalisasi Notebook", expanded=True):
                    st.image(png_norm, caption="05_normalization_comparison.png", use_container_width=True)

    with tab_denorm:
        _render_tab_denorm(h, key_prefix="vmd_gru")


def render_hasil_gru(h):
    """Render hasil Admin untuk metode GRU murni (dari file notebook)."""
    if st is None:
        return
    st.markdown(_TABS_CSS, unsafe_allow_html=True)

    tab_eval, tab_inter, tab_norm, tab_denorm = st.tabs([
        "📊 Hasil Evaluasi & Prediksi 12 Bulan",
        "📈 Data Aktual & Hasil Interpolasi",
        "📏 Sebelum & Sesudah Normalisasi",
        "🔄 Hasil Denormalisasi",
    ])

    with tab_eval:
        _render_tab_evaluasi(h, key_prefix="gru_only")

    with tab_inter:
        with st.container(border=True):
            st.markdown(
                '<div class="ct">Visualisasi Data Aktual dan Hasil Interpolasi Linear</div>'
                '<div class="cs">Perbandingan data kunjungan asli dengan hasil pengisian gap nilai 0</div>',
                unsafe_allow_html=True
            )
            col_m1, col_m2, col_m3 = st.columns(3)
            with col_m1:
                st.metric("Total Data Historis", f"{len(h['tgl_hist'])} Bulan")
            with col_m2:
                st.metric("Titik Gap (Nilai 0)", f"{len(h.get('gap_log', []))} Titik")
            with col_m3:
                rata2 = np.mean(h['aktual_hist']) if h['aktual_hist'] else 0
                st.metric("Rata-rata Pengunjung", f"{_fmt_id(rata2)} orang")
            _show_chart(fig_interpolasi(h), key="gru_fig_inter")
            if h.get("gap_log"):
                with st.expander("Lihat Rincian Titik Gap"):
                    df_gap = pd.DataFrame(h["gap_log"])
                    df_gap["date"] = pd.to_datetime(df_gap["date"]).dt.strftime("%B %Y")
                    df_gap.rename(columns={"date": "Bulan & Tahun", "visitors": "Nilai Asli",
                                           "new_value": "Hasil Imputasi Linear"}, inplace=True)
                    _show_df(df_gap, hide_index=True)
            with st.expander("Lihat Tabel Lengkap Data Historis"):
                _show_df(pd.DataFrame({
                    "Bulan": [_tgl_id(d) for d in h["tgl_hist"]],
                    "Data Asli": [_fmt_id(v) for v in h["raw_hist"]],
                    "Hasil Interpolasi": [_fmt_id(v) for v in h["aktual_hist"]],
                }), hide_index=True)

    with tab_norm:
        with st.container(border=True):
            st.markdown(
                '<div class="ct">Visualisasi Sebelum dan Sesudah Normalisasi (Min-Max 0–1)</div>'
                '<div class="cs">Transformasi skala fitur ke rentang [0, 1] agar pelatihan model GRU stabil</div>',
                unsafe_allow_html=True
            )
            png_norm = h.get("png_map", {}).get("05_normalization_comparison.png")
            if png_norm and os.path.exists(png_norm):
                with st.expander("Grafik Normalisasi Notebook", expanded=True):
                    st.image(png_norm, caption="05_normalization_comparison.png", use_container_width=True)
            comps_list = list(h.get("comps", {}).keys())
            if comps_list:
                sel = st.selectbox("Pilih Komponen:", comps_list, key="gru_sel_norm")
                _show_chart(fig_normalisasi(h, sel), key="gru_fig_norm")

    with tab_denorm:
        _render_tab_denorm(h, key_prefix="gru_only")


def render_hasil_admin(h):
    """Dispatcher: pilih render function yang sesuai berdasarkan metode."""
    if st is None:
        return
    metode = h.get("metode", "")
    if metode == "Hybrid ICEEMDAN-VMD-GRU":
        render_hasil_iceemdan_vmd_gru(h)
    elif metode == "Hybrid ICEEMDAN-GRU":
        render_hasil_iceemdan_gru(h)
    elif metode == "Hybrid VMD-GRU":
        render_hasil_vmd_gru(h)
    else:  # GRU murni
        render_hasil_gru(h)


# TAMPILAN 

In [ ]:
%%writefile app.py
import os
import io
import re
import base64
import requests
import folium
import streamlit as st
import streamlit.components.v1 as components
from html import unescape
from urllib.parse import urlparse, parse_qs, urljoin

# ============================================================
# KONFIGURASI
# ============================================================
st.set_page_config(
    page_title="Prediksi Kunjungan Wisatawan Kabupaten Bangkalan",
    page_icon="🏝️",
    layout="wide",
    initial_sidebar_state="expanded",
)

if "page" not in st.session_state:
    st.session_state.page = "landing"
if "role" not in st.session_state:
    st.session_state.role = None

ADMIN_USER = "admin"
ADMIN_PASS = "admin"
NAMA_PENGEMBANG = "Alvyan Maulana Karnawan Putra"
PETA_STYLE = "google"           # "google" atau "osm"

# Warna "biru teknik" untuk logo yang diberi "recolor": True (R, G, B)
TEKNIK_BLUE = (30, 136, 229)

ASSETS = "assets"
os.makedirs(ASSETS, exist_ok=True)

# ============================================================
# LOGO KIRI ATAS (urutan dari kiri ke kanan)
# "file"   : nama file di assets/ (dipakai lebih dulu jika ada)
# "url"    : link gambar ATAU link halaman (gambar dicari otomatis)
# "dark"   : True -> logo diberi latar biru tua gelap (untuk logo berwarna putih)
# "recolor": True -> hitam jadi putih, putih jadi biru teknik
# ============================================================
LOGOS = [
    {"file": "logo_utm.jpg", "h": 50, "dark": False, "recolor": False,
     "url": "https://www.instagram.com/p/De83c8qSJ6v/?exln=aWtueWYxaDk4ZDEw"},
    {"file": "logo_ft.png", "h": 46, "dark": False, "recolor": False,
     "url": "https://www.instagram.com/p/De83lZQSI8g/?srtk=MWg1eDc0ZjJ6dHhndA=="},
    {"file": "logo_si.png", "h": 46, "dark": False, "recolor": False,
     "url": "https://sivitas.sakera.id/images/logo-si.png"},
    {"file": "logo_bangkalan.jpg", "h": 52, "dark": False, "recolor": False,
     "url": "https://www.instagram.com/p/De832OMy3TG/?rpxt=YzY5bTdvaHVqNnlt"},
]

# ============================================================
# DATA DESTINASI
# ============================================================
DESTINASI = [
    {"nama": "Makam Syaikhona Kholil", "file": "syaikhona_kholil.jpg", "tag": "Wisata Religi",
     "icon": "🕌", "grad": "#0ea5e9,#0d9488", "color": "#ef4444",
     "url": "https://www.instagram.com/p/DXOb8I5k0WR/",
     "desk": "Makam ulama besar Madura, destinasi wisata religi utama di Bangkalan.",
     "lat": -7.0450, "lon": 112.7340},
    {"nama": "Kolla Langgundi", "file": "langgundi.jpg", "tag": "Wisata Religi",
     "icon": "🏛️", "grad": "#f59e0b,#ef4444", "color": "#f97316",
     "url": "https://blogger.googleusercontent.com/img/b/R29vZ2xl/AVvXsEh92G0yWlNc1fnYnq_qm52Dl195gR47Im6Ac5jJS47uz8aSiSq7wx7d1ojxVkuGp1ao_cyu9KeKJnkBti8n3nCErAjmT08oSEvXKNhTJRS6O9ZrIB7b6n9lpI6bzTwThx-_uWkKcd3pUKht/w640-h480-c/Wisata+Religi+Sultan+Abdul+Kadirun+Bangkalan.jpg",
     "desk": "Kawasan wisata sejarah dan budaya khas Bangkalan.",
     "lat": -7.0300, "lon": 112.7600},
    {"nama": "Makam Sunan Cendana", "file": "cendana.jpg", "tag": "Wisata Religi",
     "icon": "🌿", "grad": "#10b981,#0284c7", "color": "#22c55e",
     "url": "https://www.haibunda.com/moms-life/20210726124816-77-228535/9-potret-astana-giribangun-makam-keluarga-cendana-dilengkapi-masjid-pasar",
     "desk": "Wisata religi yang ramai dikunjungi peziarah dari berbagai daerah.",
     "lat": -7.0100, "lon": 112.7900},
    {"nama": "Makam Air Mata Ibu", "file": "air_mata_ibu.jpg", "tag": "Wisata Religi",
     "icon": "🕊️", "grad": "#8b5cf6,#ec4899", "color": "#a855f7",
     "url": "https://awsimages.detik.net.id/community/media/visual/2025/09/13/situs-makam-air-mata-ratu-ibu-1757752879199_169.png?w=1200",
     "desk": "Makam Air Mata Ibu, destinasi wisata religi dengan nilai sejarah tinggi.",
     "lat": -7.0000, "lon": 112.8000},
    {"nama": "Gunung Geger", "file": "gunung_geger.jpg", "tag": "Wisata Alam",
     "icon": "⛰️", "grad": "#22c55e,#0d9488", "color": "#0284c7",
     "url": "https://kataomed.com/wp-content/uploads/2023/03/Camping-Ground-Geger-Bintang-Photo-by-Wisatainfo-e1680515282467.jpg",
     "desk": "Destinasi wisata alam dengan jalur susur alam yang asri.",
     "lat": -7.1000, "lon": 112.9300},
]


# ============================================================
# HELPER UMUM
# ============================================================
def html(s: str):
    """Render HTML tanpa indentasi/baris kosong (hindari bug code-block markdown)."""
    s = "\n".join(l.strip() for l in s.splitlines() if l.strip())
    st.markdown(s, unsafe_allow_html=True)


def goto(page: str, role=None):
    st.session_state.page = page
    if role is not None or page in ("landing", "login"):
        st.session_state.role = role
    st.rerun()


# ============================================================
# HELPER GAMBAR
# ============================================================
UA_BROWSER = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")
UA_CRAWLER = "facebookexternalhit/1.1 (+http://www.facebook.com/externalhit_uatext.php)"


def clean_url(url: str) -> str:
    """Jika berupa link Google 'imgres', ambil alamat gambar aslinya (parameter imgurl)."""
    if url and "google." in url and "/imgres" in url:
        q = parse_qs(urlparse(url).query)
        if q.get("imgurl"):
            return q["imgurl"][0]
    return url


def looks_like_image(url: str) -> bool:
    return bool(re.search(r"\.(jpe?g|png|webp|gif)(\?|$)", url, re.I))


def sniff_mime(data: bytes):
    """Mengembalikan mime jika data benar-benar gambar, selain itu None."""
    if data[:8] == b"\x89PNG\r\n\x1a\n":
        return "image/png"
    if data[:3] == b"\xff\xd8\xff":
        return "image/jpeg"
    if data[:4] == b"RIFF" and data[8:12] == b"WEBP":
        return "image/webp"
    if data[:4] == b"GIF8":
        return "image/gif"
    return None


def shrink(data: bytes, max_side: int = 900) -> bytes:
    """Perkecil gambar. Transparansi (PNG) dipertahankan."""
    try:
        from PIL import Image
        im = Image.open(io.BytesIO(data))
        has_alpha = im.mode in ("RGBA", "LA") or "transparency" in im.info
        im = im.convert("RGBA" if has_alpha else "RGB")
        im.thumbnail((max_side, max_side))
        buf = io.BytesIO()
        if has_alpha:
            im.save(buf, format="PNG", optimize=True)
        else:
            im.save(buf, format="JPEG", quality=85)
        return buf.getvalue()
    except Exception:
        return data


def recolor_logo(data: bytes, blue=TEKNIK_BLUE) -> bytes:
    """Hitam -> putih, putih -> biru teknik (abu-abu diinterpolasi halus).
    Piksel berwarna (mis. kuning/merah) tidak diubah."""
    try:
        import numpy as np
        from PIL import Image
        im = Image.open(io.BytesIO(data)).convert("RGBA")
        a = np.array(im).astype(np.float32)
        rgb = a[..., :3]
        mx, mn = rgb.max(axis=2), rgb.min(axis=2)
        sat = (mx - mn) / (mx + 1e-6)
        lum = rgb.mean(axis=2) / 255.0
        gray = sat < 0.25
        white = np.array([255, 255, 255], dtype=np.float32)
        blu = np.array(blue, dtype=np.float32)
        t = lum[..., None]
        mixed = (1 - t) * white + t * blu
        out = a.copy()
        out[..., :3] = np.where(gray[..., None], mixed, rgb)
        buf = io.BytesIO()
        Image.fromarray(out.astype(np.uint8), "RGBA").save(buf, format="PNG")
        return buf.getvalue()
    except Exception:
        return data


def to_uri(data: bytes) -> str:
    return f"data:{sniff_mime(data)};base64,{base64.b64encode(data).decode()}"


def find_image_in_html(text: str, base: str):
    """Cari alamat gambar utama di sebuah halaman web (logo organisasi / og:image / logo)."""
    pola = [
        r'<img[^>]+src=["\']([^"\']*uploads/(?:group|organization)/[^"\']+)',
        r'<meta[^>]+property=["\']og:image(?::secure_url)?["\'][^>]+content=["\']([^"\']+)',
        r'<meta[^>]+content=["\']([^"\']+)["\'][^>]+property=["\']og:image["\']',
        r'<meta[^>]+name=["\']twitter:image["\'][^>]+content=["\']([^"\']+)',
        r'<img[^>]+src=["\']([^"\']*(?:uploads|logo)[^"\']*\.(?:png|jpe?g|webp)(?:\.crdownload)?)',
    ]
    for p in pola:
        m = re.search(p, text, re.I)
        if m:
            return urljoin(base, unescape(m.group(1)))
    return None


def download(url: str, depth: int = 0):
    """Unduh gambar. Jika URL berupa halaman web, cari gambarnya otomatis.
    Return (bytes, error)."""
    origin = "{u.scheme}://{u.netloc}/".format(u=urlparse(url))
    kandidat = [url]
    m = re.search(r"instagram\.com/(?:p|reel)/([A-Za-z0-9_-]+)", url)
    if m:   # Instagram: coba endpoint media dulu
        kandidat = [f"https://www.instagram.com/p/{m.group(1)}/media/?size=l", url]

    variasi = [
        {"User-Agent": UA_BROWSER, "Accept": "image/avif,image/webp,image/*,*/*;q=0.8"},
        {"User-Agent": UA_BROWSER, "Referer": origin, "Accept": "text/html,image/*,*/*;q=0.8"},
        {"User-Agent": UA_CRAWLER, "Accept": "text/html,*/*"},
    ]
    last = "tidak diketahui"
    for cand in kandidat:
        for h in variasi:
            try:
                r = requests.get(cand, headers=h, timeout=8, allow_redirects=True)
                if r.status_code != 200:
                    last = f"HTTP {r.status_code}"
                    continue
                if sniff_mime(r.content):
                    return r.content, None
                if depth < 1 and "text/html" in r.headers.get("Content-Type", ""):
                    img_url = find_image_in_html(r.text, cand)
                    if img_url:
                        data, err = download(img_url, depth + 1)
                        if data:
                            return data, None
                        last = err or last
                        continue
                last = "respons bukan gambar (halaman diblokir / butuh login)"
            except Exception as e:
                last = type(e).__name__
    return None, last


def get_image(file: str, url: str, resize: bool, label: str,
              recolor: bool = False, max_side: int = 900):
    """
    Urutan: (1) file lokal valid -> (2) unduh lalu simpan -> (3) link langsung ke browser -> None.
    Hasil disimpan di session_state agar tidak mengunduh ulang tiap rerun.
    """
    cache = st.session_state.setdefault("_img_cache", {})
    status = st.session_state.setdefault("_img_status", {})
    if file in cache:
        return cache[file]

    path = os.path.join(ASSETS, file)

    def selesai(data: bytes):
        if recolor:
            data = recolor_logo(data)
        cache[file] = to_uri(data)
        status[file] = {"label": label, "mode": "ok", "err": ""}
        return cache[file]

    # 1) file lokal
    if os.path.exists(path):
        with open(path, "rb") as f:
            data = f.read()
        if sniff_mime(data):
            if resize and len(data) > 400_000:
                data = shrink(data, max_side)
            return selesai(data)
        os.remove(path)   # file rusak -> hapus

    # 2) unduh dari URL
    real = clean_url(url)
    err = "URL kosong"
    if real:
        data, err = download(real)
        if data:
            if resize:
                data = shrink(data, max_side)
            with open(path, "wb") as f:
                f.write(data)
            return selesai(data)

        # 3) fallback: biarkan browser memuat (hanya jika benar-benar link gambar)
        if looks_like_image(real) and not recolor:
            cache[file] = real
            status[file] = {"label": label, "mode": "hotlink", "err": err}
            return real

    status[file] = {"label": label, "mode": "gagal", "err": err}
    return None


def reset_image_cache():
    st.session_state.pop("_img_cache", None)
    st.session_state.pop("_img_status", None)


# ============================================================
# CSS
# ============================================================
html("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&display=swap');
html, body, [class*="css"], .stApp {font-family:'Plus Jakarta Sans',sans-serif;}
.stApp {background:linear-gradient(180deg,#f1f5f9 0%,#ffffff 40%,#f8fafc 100%);}
.block-container {max-width:1150px; padding-top:1.4rem; padding-bottom:2rem;}
#MainMenu, footer, header {visibility:hidden;}

/* ---------- LOGO ATAS ---------- */
.logostrip {display:inline-flex; align-items:center; gap:18px; padding:.6rem 1.2rem;
            background:#fff; border:1px solid #e2e8f0; border-radius:18px;
            box-shadow:0 6px 18px rgba(15,23,42,.07); margin-bottom:1rem;}
.logostrip img {object-fit:contain; display:block;}
.logostrip .dk {background:linear-gradient(135deg,#0a1a33,#10294d); border-radius:12px;
                padding:6px 12px; display:flex; align-items:center;}
.logostrip .sep {width:1px; height:34px; background:#e2e8f0;}
.logofb {font-size:1.6rem; letter-spacing:6px;}

/* ---------- HEADER ---------- */
.hcard {background:#fff; border:1px solid #e2e8f0; border-radius:22px; padding:1.7rem 1.8rem;
        box-shadow:0 10px 30px rgba(15,23,42,.07); min-height:250px;}
.hcard h1 {margin:0 0 .8rem 0; font-size:2.05rem; line-height:1.2; font-weight:800; color:#0f172a;}
.hcard p {margin:0 0 1rem 0; color:#475569; font-size:.92rem; line-height:1.6;}
.chip {display:inline-block; margin:0 8px 6px 0; padding:6px 14px; border-radius:999px;
       background:#f8fafc; border:1px solid #e2e8f0; color:#475569; font-size:.78rem;}
.chip b {color:#0f172a;}

.rcard {background:rgba(255,255,255,.95); border:1px solid #e2e8f0; border-radius:22px;
        padding:1.5rem 1.6rem; box-shadow:0 10px 30px rgba(15,23,42,.07); min-height:250px;}
.rcard h4 {margin:0 0 .6rem 0; color:#0f172a; font-size:1rem; font-weight:800;}
.rcard p {margin:0 0 1rem 0; color:#475569; font-size:.86rem; line-height:1.6;}
.mini {display:grid; grid-template-columns:1fr 1fr; gap:12px;}
.mini div {border:1px solid #e2e8f0; border-radius:14px; padding:.7rem .9rem; background:#fff;}
.mini small {display:block; color:#64748b; font-size:.72rem; font-weight:600;}
.mini b {color:#0f172a; font-size:1rem;}

/* ---------- WADAH HERO (biru muda -> putih) ---------- */
.herowrap {display:grid; grid-template-columns:1.5fr 1fr; gap:1.4rem; align-items:stretch;
           padding:1.5rem; border-radius:30px; border:1px solid #bfdbfe;
           background:linear-gradient(135deg,#bae6fd 0%,#e0f2fe 45%,#ffffff 100%);
           box-shadow:0 14px 40px rgba(2,132,199,.14);}
.herowrap .hcard, .herowrap .rcard {min-height:0; height:100%; box-sizing:border-box;
           background:rgba(255,255,255,.92); border:1px solid rgba(186,230,253,.9);
           box-shadow:0 8px 24px rgba(15,23,42,.06);}
.herowrap .rcard {display:flex; flex-direction:column;}
.herowrap .rcard .mini {margin-top:auto;}
@media (max-width: 800px) {.herowrap {grid-template-columns:1fr;}}

/* ---------- SECTION ---------- */
.sec {margin:2.4rem 0 1rem 0;}
.sec h2 {margin:0; font-size:1.4rem; font-weight:800; color:#0f172a;}
.sec h2::after {content:""; display:block; width:54px; height:4px; border-radius:4px; margin-top:6px;
        background:linear-gradient(90deg,#0284c7,#0d9488);}
.sec span {color:#64748b; font-size:.86rem;}

/* ---------- PETA ---------- */
iframe {border-radius:22px; border:1px solid #e2e8f0 !important;
        box-shadow:0 10px 30px rgba(15,23,42,.10);}
.legend {display:flex; flex-wrap:wrap; gap:8px; margin-top:.8rem;}
.legend span {display:inline-flex; align-items:center; gap:6px; background:#fff;
       border:1px solid #e2e8f0; border-radius:999px; padding:5px 13px; font-size:.78rem;
       color:#334155; font-weight:600;}
.legend i {width:10px; height:10px; border-radius:50%; display:inline-block;}

/* ---------- KARTU DESTINASI ---------- */
.dest {background:#fff; border:1px solid #e2e8f0; border-radius:22px; overflow:hidden;
       box-shadow:0 4px 14px rgba(15,23,42,.06); transition:.3s; margin-bottom:1rem;}
.dest:hover {transform:translateY(-6px); box-shadow:0 18px 36px rgba(2,132,199,.18);}
.dest .pic {height:175px; display:flex; align-items:center; justify-content:center;
       font-size:3.2rem; position:relative; overflow:hidden;}
.dest .pic img.pimg {position:absolute; inset:0; width:100%; height:100%; object-fit:cover;}
.dest .tag {position:absolute; top:12px; left:12px; z-index:2; background:rgba(255,255,255,.92);
       color:#0369a1; font-size:.7rem; font-weight:700; padding:4px 10px; border-radius:999px;}
.dest .body {padding:1rem 1.1rem 1.2rem 1.1rem;}
.dest h4 {margin:0 0 .3rem 0; color:#0f172a; font-size:1rem;}
.dest p {margin:0; color:#64748b; font-size:.82rem;}

/* ---------- CTA & LOGIN ---------- */
.cta {text-align:center; border-radius:26px; padding:2.4rem 1.5rem; margin-top:2.4rem; color:#fff;
      background:linear-gradient(135deg,#0f172a 0%,#0369a1 60%,#0d9488 100%);
      box-shadow:0 18px 40px rgba(15,23,42,.25);}
.cta h3 {margin:0 0 .4rem 0; color:#fff; font-size:1.5rem; font-weight:800;}
.cta p {margin:0; color:rgba(255,255,255,.85); font-size:.92rem;}

.lcard {background:#fff; border:1px solid #e2e8f0; border-radius:22px; padding:1.5rem;
        box-shadow:0 10px 28px rgba(15,23,42,.08); margin-bottom:.8rem;}
.lcard h3 {margin:0 0 .3rem 0; color:#0f172a;}
.lcard p {margin:0; color:#64748b; font-size:.88rem;}

.dash {background:linear-gradient(135deg,#0369a1,#0d9488); color:#fff; border-radius:22px;
       padding:1.4rem 1.6rem; box-shadow:0 14px 34px rgba(2,132,199,.25); margin-bottom:1rem;}
.dash h2 {margin:0; color:#fff; font-size:1.5rem; font-weight:800;}
.dash span {color:rgba(255,255,255,.88); font-size:.88rem;}

div.stButton > button, div.stFormSubmitButton > button {
    background:linear-gradient(90deg,#0284c7,#0d9488); color:#fff; border:none;
    border-radius:14px; padding:.75rem 1rem; font-weight:700; width:100%;
    box-shadow:0 8px 20px rgba(2,132,199,.30); transition:.2s;}
div.stButton > button:hover, div.stFormSubmitButton > button:hover {
    color:#fff; transform:translateY(-2px); box-shadow:0 12px 26px rgba(2,132,199,.40);}
.footer {text-align:center; color:#94a3b8; font-size:.78rem; margin-top:2.2rem;}
</style>
""")


# ============================================================
# KOMPONEN
# ============================================================
def logo_strip() -> str:
    items = []
    for lg in LOGOS:
        src = get_image(lg["file"], lg["url"], True, "Logo " + lg["file"],
                        recolor=lg.get("recolor", False), max_side=500)
        if not src:
            continue
        img = (f'<img src="{src}" alt="" referrerpolicy="no-referrer" '
               f'style="height:{lg["h"]}px;"/>')
        items.append(f'<div class="dk">{img}</div>' if lg["dark"] else img)
    if not items:
        return '<div class="logostrip"><span class="logofb">🎓 🏛️ 🌴</span></div>'
    return '<div class="logostrip">' + '<div class="sep"></div>'.join(items) + "</div>"


def render_peta():
    if PETA_STYLE == "google":
        m = folium.Map(location=[-7.05, 112.83], zoom_start=10, control_scale=True,
                       tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}&hl=id",
                       attr="Google Maps")
    else:
        m = folium.Map(location=[-7.05, 112.83], zoom_start=10, tiles="OpenStreetMap",
                       control_scale=True)

    for d in DESTINASI:
        pin = f"""
        <div style="position:relative; width:30px; height:42px; transform:translate(-15px,-40px);">
          <svg viewBox="0 0 30 42" width="30" height="42" xmlns="http://www.w3.org/2000/svg"
               style="filter:drop-shadow(0 3px 3px rgba(0,0,0,.35));">
            <path d="M15 0C6.7 0 0 6.7 0 15c0 11 15 27 15 27s15-16 15-27C30 6.7 23.3 0 15 0z"
                  fill="{d['color']}" stroke="white" stroke-width="2"/>
            <circle cx="15" cy="15" r="6" fill="white"/>
          </svg>
        </div>
        """
        folium.Marker(
            location=[d["lat"], d["lon"]],
            icon=folium.DivIcon(html=pin, icon_size=(0, 0)),
            tooltip=folium.Tooltip(
                f"<b>{d['nama']}</b>", permanent=True, direction="right",
                style="background:white;color:#0f172a;border-radius:8px;"
                      "border:1px solid #e2e8f0;padding:3px 8px;font-size:12px;"
                      "box-shadow:0 2px 6px rgba(0,0,0,.2);margin-left:10px;"),
            popup=folium.Popup(
                f"<b>{d['nama']}</b><br><span style='color:#64748b'>{d['tag']}</span>",
                max_width=220),
        ).add_to(m)

    m.fit_bounds([[d["lat"], d["lon"]] for d in DESTINASI], padding=(40, 40))
    peta_html = m.get_root().render()
    if hasattr(st, "iframe"):          # Streamlit baru (components.html sudah dihapus)
        st.iframe(peta_html, height=480)
    else:
        components.html(peta_html, height=480)

    legend = "".join(
        f'<span><i style="background:{d["color"]}"></i>{d["nama"]}</span>' for d in DESTINASI)
    html(f'<div class="legend">{legend}</div>')


def dest_card(d: dict) -> str:
    src = get_image(d["file"], d["url"], True, d["nama"])
    grad = f'linear-gradient(135deg,{d["grad"]})'
    img = (f'<img class="pimg" src="{src}" alt="" referrerpolicy="no-referrer"/>'
           if src else "")
    return f"""
    <div class="dest">
    <div class="pic" style="background-image:{grad};"><span class="tag">{d['tag']}</span>{d['icon']}{img}</div>
    <div class="body"><h4>{d['nama']}</h4><p>{d['desk']}</p></div>
    </div>
    """


def section(title: str, sub: str = ""):
    html(f'<div class="sec"><h2>{title}</h2><span>{sub}</span></div>')


def panel_status_gambar():
    """Panel info + upload manual untuk gambar/logo yang belum termuat dari server."""
    status = st.session_state.get("_img_status", {})
    masalah = {f: s for f, s in status.items() if s["mode"] != "ok"}
    if not masalah:
        return

    with st.expander(f"ℹ️ {len(masalah)} gambar belum diunduh server (klik untuk upload manual)"):
        st.caption(
            "Instagram dan beberapa situs memblokir unduhan otomatis. Simpan fotonya dari "
            "browser/HP Anda, lalu upload di bawah. Hasil upload tersimpan di folder assets/.")
        for file, s in masalah.items():
            st.markdown(f"**{s['label']}** · status: `{s['mode']}` · alasan: `{s['err']}`")
            up = st.file_uploader(f"Upload untuk {file}", type=["jpg", "jpeg", "png", "webp"],
                                  key=f"up_{file}")
            if up is not None:
                data = up.getvalue()
                if sniff_mime(data):
                    with open(os.path.join(ASSETS, file), "wb") as f:
                        f.write(data)
                    reset_image_cache()
                    st.rerun()
                else:
                    st.error("File bukan gambar yang valid.")
        if st.button("🔄 Coba muat ulang semua gambar", key="btn_retry"):
            reset_image_cache()
            st.rerun()


# ============================================================
# HALAMAN: LANDING
# ============================================================
def landing_page():
    html(logo_strip())

    # ---------- HERO (2 card dalam wadah biru muda -> putih) ----------
    html(f"""
    <div class="herowrap">
    <div class="hcard">
    <h1>Aplikasi Prediksi Jumlah Kunjungan Wisatawan<br>Di Kabupaten Bangkalan</h1>
    <p>Dikembangkan oleh <b>{NAMA_PENGEMBANG}</b> sebagai Tugas Akhir S1 Sistem Informasi.
    Sistem memanfaatkan metode hybrid <b>ICEEMDAN-VMD-GRU</b> untuk membantu analisis
    tren kunjungan wisata.</p>
    <span class="chip">Metode: <b>ICEEMDAN-VMD</b></span>
    <span class="chip">Model: <b>GRU</b></span>
    <span class="chip">Output: <b>Prediksi Kunjungan</b></span>
    </div>
    <div class="rcard">
    <h4>Ringkasan Metode</h4>
    <p>Kombinasi <b>ICEEMDAN</b> dan <b>Variational Mode Decomposition (VMD)</b> untuk
    dekomposisi sinyal dan <b>Gated Recurrent Unit (GRU)</b> untuk pemodelan deret waktu,
    sehingga prediksi lebih stabil terhadap noise dan pola musiman.</p>
    <div class="mini">
    <div><small>Cakupan</small><b>Kab. Bangkalan</b></div>
    <div><small>Destinasi</small><b>{len(DESTINASI)} Lokasi</b></div>
    </div>
    </div>
    </div>
    """)

    # ---------- PETA ----------
    section("Wilayah Cakupan Kabupaten Bangkalan", "Pin berwarna menandai lokasi objek wisata")
    render_peta()

    # ---------- DESTINASI ----------
    section("Destinasi Wisata yang Diprediksi", f"{len(DESTINASI)} lokasi wisata ditampilkan")
    r1 = st.columns(3, gap="medium")
    for col, d in zip(r1, DESTINASI[:3]):
        with col:
            html(dest_card(d))
    r2 = st.columns(3, gap="medium")
    for col, d in zip(r2, DESTINASI[3:]):
        with col:
            html(dest_card(d))

    # ---------- CTA ----------
    html("""
    <div class="cta">
    <h3>Siap mulai analisis prediksi?</h3>
    <p>Silakan login sebagai Admin atau User untuk mengakses fitur peramalan kunjungan wisatawan.</p>
    </div>
    """)
    st.write("")
    _, mid, _ = st.columns([1, 1.2, 1])
    with mid:
        if st.button("🔐 Login Admin atau User", key="btn_login"):
            goto("login")

    html(f'<div class="footer">© 2026 {NAMA_PENGEMBANG} — Aplikasi Prediksi Kunjungan '
         'Wisatawan Kabupaten Bangkalan, Tugas Akhir. All rights reserved.</div>')

    panel_status_gambar()


# ============================================================
# HALAMAN: LOGIN
# ============================================================
def login_page():
    html("""
    <div class="hcard" style="text-align:center; min-height:0;">
    <h1 style="margin:0;">Selamat Datang 👋</h1>
    <p style="margin:.5rem 0 0 0;">Pilih cara masuk untuk melanjutkan ke aplikasi peramalan.</p>
    </div>
    """)
    st.write("")
    cu, ca = st.columns(2, gap="large")

    # ----- USER: langsung masuk tanpa input -----
    with cu:
        html("""
        <div class="lcard"><h3>👤 User</h3>
        <p>Masuk langsung tanpa username dan password untuk melihat serta menjalankan peramalan.</p></div>
        """)
        if st.button("Masuk sebagai User", key="btn_user"):
            goto("app_user", role="User")

    # ----- ADMIN: username & password -----
    with ca:
        html("""
        <div class="lcard"><h3>🛠️ Admin</h3>
        <p>Masuk dengan akun admin untuk mengelola data dan model.</p></div>
        """)
        with st.form("form_admin"):
            u = st.text_input("Username")
            p = st.text_input("Password", type="password")
            ok = st.form_submit_button("Masuk sebagai Admin")
        if ok:
            if u == ADMIN_USER and p == ADMIN_PASS:
                goto("app_admin", role="Admin")
            else:
                st.error("Username atau password salah.")

    st.write("")
    _, mid, _ = st.columns([1, 1, 1])
    with mid:
        if st.button("⬅️ Kembali ke Beranda", key="btn_back"):
            goto("landing")


# ============================================================
# HALAMAN: USER  (Dashboard Prediksi)
# ============================================================
import inspect
import pandas as pd
import plotly.graph_objects as go
import forecasting_engine as fe

BULAN_PENDEK = ["Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]

DASH_CSS = """
<style>
/* header Streamlit tetap ada (tombol sidebar di HP), hanya toolbar disembunyikan */
header[data-testid="stHeader"] {visibility:visible !important; background:transparent !important;}
[data-testid="stToolbar"], [data-testid="stDecoration"] {display:none !important;}
.block-container {max-width:1250px;}

/* ---------- SIDEBAR ---------- */
section[data-testid="stSidebar"] {background:#f1f5f9; border-right:1px solid #e2e8f0;}
section[data-testid="stSidebar"] .block-container {padding-top:1.5rem;}
.sb-h {font-weight:800; color:#0f172a; font-size:.98rem; margin:.2rem 0 .6rem 0;}
.sb-model {background:#dbeafe; color:#1e40af; border-radius:10px; padding:.55rem .8rem;
           font-size:.78rem; margin-top:.4rem;}
section[data-testid="stSidebar"] label p {font-size:.8rem; color:#334155; font-weight:600;}
section[data-testid="stSidebar"] [data-testid="stFileUploaderDropzone"] {background:#fff;}
section[data-testid="stSidebar"] hr {margin:1rem 0;}
div.stButton > button:disabled {opacity:.45; cursor:not-allowed; box-shadow:none;}

/* ---------- HEADER DASHBOARD ---------- */
.dwrap {position:relative; margin-bottom:1.1rem;}
.dhead {max-width:720px; margin:0 auto; padding:1.1rem 1.6rem 1.3rem 1.6rem; border-radius:24px;
        background:linear-gradient(135deg,#bae6fd 0%,#eff9ff 55%,#ffffff 100%);
        border:1px solid #dbeafe; box-shadow:0 12px 30px rgba(2,132,199,.14);}
.dlogos {display:inline-flex; align-items:center; gap:12px; background:#fff; border-radius:14px;
         padding:.4rem .8rem; box-shadow:0 3px 10px rgba(15,23,42,.08); margin-bottom:.8rem;}
.dlogos img {display:block; object-fit:contain;}
.dhead h1 {margin:0; font-size:1.7rem; font-weight:800; color:#0f172a; line-height:1.25;}
.dhead h2 {margin:.1rem 0 0 0; font-size:1.45rem; font-weight:800; color:#0f172a; line-height:1.25;}
.dbadge {position:absolute; right:0; top:0; background:#fff; border-radius:14px; padding:.45rem .6rem;
         box-shadow:0 3px 10px rgba(15,23,42,.10);}
.dbadge img {display:block; object-fit:contain;}
@media (max-width: 900px) {.dbadge {display:none;}}

/* ---------- INFO & PLACEHOLDER ---------- */
.dinfo {background:#e8f1fd; color:#1e3a8a; border-radius:10px; padding:.75rem 1rem; font-size:.84rem;
        margin:.4rem 0 1rem 0;}
.sk {background:#f1f5f9; border-radius:18px;}
.sk-row {display:grid; grid-template-columns:1fr 1fr; gap:1rem; margin-bottom:1rem;}

/* ---------- KARTU METRIK ---------- */
.mcard {background:#fff; border:1px solid #e2e8f0; border-radius:18px; padding:1rem 1.3rem;
        box-shadow:0 6px 18px rgba(15,23,42,.06); margin-bottom:1rem;}
.mcard .ml {color:#64748b; font-size:.78rem; font-weight:700; letter-spacing:.3px;}
.mcard .mv {color:#0f172a; font-size:2rem; font-weight:800; line-height:1.2; margin:.15rem 0;}
.mcard .ms {color:#64748b; font-size:.78rem;}
.mcard.a {border-top:4px solid #0284c7;}
.mcard.b {border-top:4px solid #0d9488;}

/* ---------- KARTU GRAFIK ---------- */
div[data-testid="stVerticalBlockBorderWrapper"] {border-radius:18px;}
.ct {font-weight:800; color:#0f172a; font-size:1.05rem; margin:0;}
.cs {color:#64748b; font-size:.8rem; margin:.1rem 0 .4rem 0;}
.dfooter {text-align:center; color:#64748b; font-size:.74rem; margin-top:2.2rem; padding-top:.9rem;
          border-top:1px solid #e2e8f0;}
</style>
"""


def _fmt_id(x, dec=0):
    s = f"{x:,.{dec}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")


def _tgl_id(iso):
    d = pd.Timestamp(iso)
    return f"{BULAN_PENDEK[d.month - 1]} {d.year}"


def _show(obj, kind="plot", **kw):
    """Kompatibel lintas versi Streamlit (width='stretch' vs use_container_width)."""
    fn = st.plotly_chart if kind == "plot" else st.dataframe
    if "width" in inspect.signature(fn).parameters:
        return fn(obj, width="stretch", **kw)
    return fn(obj, use_container_width=True, **kw)


def _logo_img(lg, h=None):
    src = get_image(lg["file"], lg["url"], True, "Logo " + lg["file"],
                    recolor=lg.get("recolor", False), max_side=500)
    if not src:
        return ""
    return f'<img src="{src}" alt="" referrerpolicy="no-referrer" style="height:{h or lg["h"]}px;"/>'


def dash_header():
    kiri = "".join(_logo_img(lg) for lg in LOGOS[:-1])
    kanan = _logo_img(LOGOS[-1], 46)
    html(f"""
    <div class="dwrap">
    <div class="dhead">
    <div class="dlogos">{kiri or "🎓 🏛️"}</div>
    <h1>Dashboard Prediksi</h1>
    <h2>Analisis Data Kunjungan dan Peramalan Masa Depan</h2>
    </div>
    <div class="dbadge">{kanan or "🏛️"}</div>
    </div>
    """)


def dash_footer():
    html(f'<div class="dfooter">© 2026 {NAMA_PENGEMBANG} - Tugas Akhir S1 Sistem Informasi UTM. '
         'All rights reserved.</div>')


def dash_info(teks):
    html(f'<div class="dinfo">{teks}</div>')


def dash_skeleton():
    html("""
    <div class="sk-row"><div class="sk" style="height:96px"></div><div class="sk" style="height:96px"></div></div>
    <div class="sk" style="height:260px"></div>
    """)


def _metric_card(cls, label, nilai, sub):
    return (f'<div class="mcard {cls}"><div class="ml">{label}</div>'
            f'<div class="mv">{nilai}</div><div class="ms">{sub}</div></div>')


def _layout(fig, h=380):
    fig.update_layout(
        height=h, margin=dict(l=8, r=8, t=8, b=8), plot_bgcolor="#ffffff", paper_bgcolor="#ffffff",
        font=dict(family="Plus Jakarta Sans, sans-serif", size=12, color="#334155"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        hovermode="x unified",
        xaxis=dict(showgrid=False, tickformat="%Y", dtick="M12", linecolor="#cbd5e1"),
        yaxis=dict(title="Jumlah Pengunjung", gridcolor="#e2e8f0", tickformat=",.0f", zeroline=False),
    )
    return fig


def fig_aktual_prediksi(h):
    tgl = pd.to_datetime(h["tgl_hist"])
    tgl_uji = pd.to_datetime(h["tgl_uji"])
    fig = go.Figure()
    fig.add_vrect(x0=tgl_uji[0], x1=tgl_uji[-1], fillcolor="rgba(249,115,22,0.07)", line_width=0,
                  annotation_text="Periode uji", annotation_position="top left",
                  annotation_font=dict(size=11, color="#c2410c"))
    fig.add_trace(go.Scatter(
        x=tgl, y=h["aktual_hist"], name="Aktual", mode="lines",
        line=dict(color="#0284c7", width=2.4),
        customdata=[_tgl_id(d.strftime("%Y-%m-%d")) for d in tgl],
        hovertemplate="Aktual: %{y:,.0f}<extra></extra>"))
    fig.add_trace(go.Scatter(
        x=tgl_uji, y=h["prediksi_uji"], name="Prediksi", mode="lines+markers",
        line=dict(color="#f97316", width=2.4, dash="dash"), marker=dict(size=6),
        hovertemplate="Prediksi: %{y:,.0f}<extra></extra>"))
    return _layout(fig)


def fig_masa_depan(h):
    n_tampil = 36
    tgl = pd.to_datetime(h["tgl_hist"])[-n_tampil:]
    akt = h["aktual_hist"][-n_tampil:]
    tgl_f = pd.to_datetime(h["tgl_fut"])
    fig = go.Figure()
    fig.add_vrect(x0=tgl_f[0], x1=tgl_f[-1], fillcolor="rgba(13,148,136,0.07)", line_width=0,
                  annotation_text="12 bulan ke depan", annotation_position="top left",
                  annotation_font=dict(size=11, color="#0f766e"))
    fig.add_trace(go.Scatter(x=tgl, y=akt, name="Data historis", mode="lines",
                             line=dict(color="#0284c7", width=2.4),
                             hovertemplate="Aktual: %{y:,.0f}<extra></extra>"))
    # garis penghubung titik terakhir historis -> prediksi pertama
    fig.add_trace(go.Scatter(x=[tgl[-1], tgl_f[0]], y=[akt[-1], h["prediksi_fut"][0]],
                             mode="lines", showlegend=False, hoverinfo="skip",
                             line=dict(color="#0d9488", width=1.6, dash="dot")))
    fig.add_trace(go.Scatter(x=tgl_f, y=h["prediksi_fut"], name="Prediksi 12 bulan",
                             mode="lines+markers", line=dict(color="#0d9488", width=2.6, dash="dash"),
                             marker=dict(size=7),
                             hovertemplate="Prediksi: %{y:,.0f}<extra></extra>"))
    fig.update_xaxes(tickformat="%b %Y", dtick="M6")
    return _layout(fig)


def render_hasil(h):
    c1, c2 = st.columns(2, gap="medium")
    with c1:
        html(_metric_card("a", "MAE (Mean Absolute Error)", _fmt_id(h["mae"], 2),
                          "rata-rata selisih absolut aktual dan prediksi (pengunjung)"))
    with c2:
        html(_metric_card("b", "MAPE (Mean Absolute Percentage Error)", _fmt_id(h["mape"], 2) + "%",
                          "rata-rata persentase kesalahan prediksi"))

    with st.container(border=True):
        html(f'<div class="ct">Visualisasi Data Aktual dan Prediksi</div>'
             f'<div class="cs">Evaluasi pada periode uji {_tgl_id(h["tgl_uji"][0])} – '
             f'{_tgl_id(h["tgl_uji"][-1])} ({len(h["tgl_uji"])} bulan)</div>')
        _show(fig_aktual_prediksi(h), "plot", key="fig_aktual")

    st.write("")
    with st.container(border=True):
        html(f'<div class="ct">Prediksi Kunjungan 12 Bulan ke Depan</div>'
             f'<div class="cs">{_tgl_id(h["tgl_fut"][0])} – {_tgl_id(h["tgl_fut"][-1])}</div>')
        _show(fig_masa_depan(h), "plot", key="fig_future")
        tabel = pd.DataFrame({
            "Bulan": [_tgl_id(d) for d in h["tgl_fut"]],
            "Prediksi Pengunjung": [_fmt_id(v) for v in h["prediksi_fut"]],
        })
        with st.expander("Lihat tabel prediksi"):
            _show(tabel, "df", hide_index=True)

    if h.get("n_param_default", 0) > 0:
        st.caption("Catatan: pola data yang diunggah berbeda dari data acuan model, sehingga "
                   f"{h['n_param_default']} komponen memakai pengaturan standar. Hasil dapat "
                   "berbeda dari data acuan.")


def user_page():
    if st.session_state.role != "User":
        goto("login")

    html(DASH_CSS)
    names = [d["nama"] for d in DESTINASI if d["nama"] in fe.CONFIG]

    # ---------------- SIDEBAR ----------------
    with st.sidebar:
        html('<div class="sb-h">Konfigurasi Prediksi</div>')
        dest = st.radio("Pilih Parameter & Model Destinasi", names, key="dash_dest",
                        help="Pilih destinasi wisata yang akan diprediksi.")
        html(f'<div class="sb-model">Model Dipilih: <b>{dest}</b></div>')
        st.divider()
        html('<div class="sb-h">Upload Dataset</div>')
        up = st.file_uploader("Upload Data (.xlsx atau .csv)", type=["xlsx", "csv"],
                              key="dash_up")
        st.divider()
        html('<div class="sb-h">Proses</div>')
        go_btn = st.button("Mulai Prediksi", key="dash_go", disabled=(up is None))
        st.divider()
        if st.button("« Kembali ke Beranda", key="dash_back"):
            goto("landing")

    # ---------------- HALAMAN UTAMA ----------------
    dash_header()

    if up is None:
        dash_info("Silakan unggah (upload) file dataset Excel/CSV melalui sidebar di sebelah "
                  "kiri untuk memulai analisis.")
        dash_skeleton()
        dash_footer()
        return

    data_bytes = up.getvalue()
    try:
        data_clean, info = fe.siapkan_data(fe.baca_dataset(data_bytes, up.name, destinasi=dest))
    except Exception as e:
        st.error(f"Dataset tidak dapat dibaca: {e}")
        dash_footer()
        return

    key = fe.kunci_cache(dest, data_bytes)
    clean_key = fe.kunci_cache(
        dest,
        pd.util.hash_pandas_object(
            data_clean[["date", "visitors"]].reset_index(drop=True), index=False
        ).values.tobytes()
    )

    if go_btn:
        hasil = fe.muat_cache(key) or fe.muat_cache(clean_key)
        if hasil is None or (metode == "Hybrid ICEEMDAN-VMD-GRU" and "ice_data" not in hasil):
            bar = st.progress(0.0, text="Memulai proses...")
            try:
                hasil = fe.jalankan_prediksi(
                    dest, data_clean, progress=lambda f, t: bar.progress(min(max(f, 0.0), 1.0), text=t))
            except Exception as e:
                bar.empty()
                st.error(f"Proses gagal dijalankan: {e}")
                dash_footer()
                return
            bar.empty()
            fe.simpan_cache(key, hasil)
            fe.simpan_cache(clean_key, hasil)
        st.session_state["dash_result"] = {"key": key, "hasil": hasil}

    res = st.session_state.get("dash_result")
    if res and res["key"] == key:
        render_hasil(res["hasil"])
    else:
        periode = f'{_tgl_id(info["awal"].strftime("%Y-%m-%d"))} – {_tgl_id(info["akhir"].strftime("%Y-%m-%d"))}'
        dash_info(f'Dataset <b>{up.name}</b> diterima ({info["n"]} bulan, {periode}). '
                  'Klik <b>Mulai Prediksi</b> pada sidebar untuk menjalankan analisis.')
        dash_skeleton()
    dash_footer()


# ============================================================
# HALAMAN: ADMIN (Dashboard sama seperti User + pilihan metode)
# ============================================================
import admin_methods as am


def admin_page():
    if st.session_state.role != "Admin":
        goto("login")

    html(DASH_CSS)
    names = [d["nama"] for d in DESTINASI if d["nama"] in fe.CONFIG]

    # ---------------- SIDEBAR ----------------
    with st.sidebar:
        html('<div class="sb-h">Konfigurasi Prediksi</div>')
        dest = st.radio("Pilih Parameter & Model Destinasi", names, key="adm_dest",
                        help="Pilih destinasi wisata yang akan diprediksi.")
        st.write("")
        metode = st.radio("Pilih Metode", am.METODE, key="adm_metode",
                          help="Pilih metode peramalan yang akan dijalankan.")
        html(f'<div class="sb-model">Model Dipilih: <b>{dest}</b><br>'
             f'Metode: <b>{metode}</b></div>')
        st.divider()
        html('<div class="sb-h">Upload Dataset</div>')
        up = st.file_uploader("Upload Data (.xlsx atau .csv)", type=["xlsx", "csv"],
                              key="adm_up")
        st.divider()
        html('<div class="sb-h">Proses</div>')
        go_btn = st.button("Mulai Prediksi", key="adm_go", disabled=(up is None))
        st.divider()
        if st.button("« Keluar", key="adm_back"):
            goto("landing")

    # ---------------- HALAMAN UTAMA ----------------
    dash_header()

    if up is None:
        dash_info("Silakan unggah (upload) file dataset Excel/CSV melalui sidebar di sebelah "
                  "kiri untuk memulai analisis.")
        dash_skeleton()
        dash_footer()
        return

    data_bytes = up.getvalue()
    try:
        data_clean, info = fe.siapkan_data(fe.baca_dataset(data_bytes, up.name, destinasi=dest))
    except Exception as e:
        st.error(f"Dataset tidak dapat dibaca: {e}")
        dash_footer()
        return
    # kunci cache: hybrid ICEEMDAN-VMD-GRU memakai kunci yang sama dengan User
    nama_kunci = dest if metode == am.METODE[0] else f"{dest}|{metode}"
    key = fe.kunci_cache(nama_kunci, data_bytes)
    clean_key = fe.kunci_cache(
        nama_kunci,
        pd.util.hash_pandas_object(
            data_clean[["date", "visitors"]].reset_index(drop=True), index=False
        ).values.tobytes()
    )
    state_key = f"{key}|{metode}"

    if go_btn:
        hasil = fe.muat_cache(key) or fe.muat_cache(clean_key)
        if hasil is None or (metode == "Hybrid ICEEMDAN-VMD-GRU" and "ice_data" not in hasil):
            bar = st.progress(0.0, text="Memulai proses...")
            try:
                hasil = am.jalankan_metode(
                    dest, data_clean, metode,
                    progress=lambda f, t: bar.progress(min(max(f, 0.0), 1.0), text=t))
            except Exception as e:
                bar.empty()
                st.error(f"Proses gagal dijalankan: {e}")
                dash_footer()
                return
            bar.empty()
            fe.simpan_cache(key, hasil)
            fe.simpan_cache(clean_key, hasil)
        st.session_state["adm_result"] = {"key": state_key, "hasil": hasil}

    res = st.session_state.get("adm_result")
    if res and res["key"] == state_key:
        dash_info(f"Hasil metode <b>{metode}</b> untuk destinasi <b>{dest}</b>.")
        am.render_hasil_admin(res["hasil"])
    else:
        periode = (f'{_tgl_id(info["awal"].strftime("%Y-%m-%d"))} – '
                   f'{_tgl_id(info["akhir"].strftime("%Y-%m-%d"))}')
        dash_info(f'Dataset <b>{up.name}</b> diterima ({info["n"]} bulan, {periode}). '
                  f'Metode terpilih: <b>{metode}</b>. '
                  'Klik <b>Mulai Prediksi</b> pada sidebar untuk menjalankan analisis.')
        dash_skeleton()
    dash_footer()


# ============================================================
# ROUTER
# ============================================================
page = st.session_state.page
if page == "landing":
    landing_page()
elif page == "login":
    login_page()
elif page == "app_user":
    user_page()
elif page == "app_admin":
    admin_page()
else:
    landing_page()


Overwriting app.py


In [30]:
import subprocess
subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true"]
)

<Popen: returncode: None args: ['streamlit', 'run', 'app.py', '--server.port...>

In [31]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8501



2026-10-31T14:01:22Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-31T14:01:22Z INF Requesting new quick Tunnel on trycloudflare.com...

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.79.89.238:8501



2026-10-31 14:01:23.913 Uvicorn server started on :::8501


2026-10-31T14:01:26Z INF +--------------------------------------------------------------------------------------------+
2026-10-31T14:01:26Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-31T14:01:26Z INF |  https://discount-villas-shopping-joan.trycloudflare.com                                   |
2026-10-31T14:01:26Z INF +--------------------------------------------------------------------------------------------+
2026-10-31T14:01:26Z INF Cannot determine default configuration path. No file [config.yml config.yaml] in [~/.cloudflared ~/.cloudflare-warp ~/cloudflare-warp /etc/cloudflared /usr/local/etc/cloudflared]
2026-10-31T14:01:26Z INF Version 2026.10.7 (Checksum 1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872)
2026-10-31T14:01:26Z INF GOOS: linux, GOVersion: go1.27.3, GoArch: amd64
2026-10-31T14:01:26Z INF Settings: map[ha-connections:1 protocol:quic url:http://localhost:8501]
2026-10-31T14:01:26Z INF cl

2026-10-31 14:02:33.156225: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


  Stopping...
^C
2026-10-31T14:19:11Z INF Initiating graceful shutdown due to signal interrupt ...
2026-10-31T14:19:11Z INF Retrying connection in up to 2s connIndex=0 event=0 ip=198.41.192.227
2026-10-31T14:19:11Z ERR Connection terminated connIndex=0
2026-10-31T14:19:11Z ERR no more connections active and exiting
2026-10-31T14:19:11Z INF Tunnel server stopped
2026-10-31T14:19:11Z INF Metrics server stopped
